# MedMNIST v2 ResNet baselines — independent replication (ReScience)

**One self-contained notebook** that produces every result and figure for the paper.
It writes our `src/` package to disk from embedded cells (no GitHub clone needed),
then runs one of three modes set in the **CONFIG** cell:

- `MODE="baselines"` — train the tiered run matrix (resumable across Kaggle's ~9h cap).
- `MODE="extensions"` — per-class analysis, bias mitigation, lightweight variant (DermaMNIST).
- `MODE="report"` — aggregate all runs → comparison table + every figure.

**Scope (compute-minimal slice of Table 3).** The cheapest route to a *solid*
partial replication is breadth at 28×28, not depth at 224. So we replicate
**ResNet-18 @ 28 across all twelve MedMNIST2D datasets** and keep the **full
four-config matrix (R18/R50 × 28/224) on DermaMNIST**:

- **Tier 1 — DermaMNIST (primary):** ResNet-18 and ResNet-50, sizes 28 and 224, seeds 0/1/2. 12 runs.
- **Tier 2 — small/medium datasets, R18 @ 28, seeds 0/1/2:** Retina, Breast, Pneumonia, Blood, OrganA, OrganC, OrganS. 21 runs.
- **Tier 3 — large datasets, R18 @ 28, seed 0 only:** Tissue, OCT, Path, Chest. 4 runs. Single-seed for compute reasons (stable at that data scale); disclosed in the writeup.

**~37 runs total ≈ 12–15 GPU-hours on a P100.** Tier 3's four large datasets at
a single seed dominate the cost; drop one or two if you're tight and note it.

**Hard rule:** nothing here is copied from `MedMNIST/experiments`. We use the
`medmnist` PyPI package only for (a) the standardized dataset loaders and
(b) `medmnist.Evaluator` as a metric oracle. Everything else is our own code.

### Kaggle workflow (because ~37 runs won't fit in one 9h session)
1. Turn **Internet ON** (Settings) so the datasets can download from Zenodo, and **GPU** on (P100).
2. Run `MODE="baselines"` with a `TIERS`/`MAX_MINUTES` you can finish. **Save Version** (commit).
3. Next session: **Add data → your previous notebook output**, set `PREV_RESULTS` to it, run again — finished runs skip, partial runs resume from `last.pth`.
4. When all needed runs exist: run `MODE="extensions"` once, then `MODE="report"` once.

## 1. Install deps + write the `src/` package

In [ ]:
# medmnist is the only missing dep on Kaggle (torch/torchvision/sklearn/matplotlib preinstalled).
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist==3.0.2"], check=False)


In [ ]:
SRC_B64 = {
    '__init__.py': 'IiIiSW5kZXBlbmRlbnQgcmVpbXBsZW1lbnRhdGlvbiBvZiB0aGUgTWVkTU5JU1QgdjIgUmVzTmV0IGJhc2VsaW5lcy4KCk5vdGhpbmcgaW4gdGhpcyBwYWNrYWdlIGlzIGltcG9ydGVkIG9yIGFkYXB0ZWQgZnJvbSBgYE1lZE1OSVNUL2V4cGVyaW1lbnRzYGAuClRoZSBvbmx5IE1lZE1OSVNUIGNvZGUgdXNlZCBpcyB0aGUgYGBtZWRtbmlzdGBgIFB5UEkgcGFja2FnZTogaXRzIGRhdGFzZXQKbG9hZGVycyAoZm9yIHRoZSBzdGFuZGFyZGl6ZWQgZGF0YSkgYW5kIGBgRXZhbHVhdG9yYGAgKGFzIGEgbWV0cmljIG9yYWNsZSkuCiIiIgoKX19hbGxfXyA9IFsibWV0cmljcyIsICJtb2RlbHMiLCAiZGF0YSIsICJ0cmFpbiIsICJldmFsdWF0ZSIsICJydW4iLCAiYWdncmVnYXRlIiwKICAgICAgICAgICAiZXh0ZW5zaW9ucyJdCg==',
    'aggregate.py': 'IiIiQWdncmVnYXRlIHBlci1ydW4gYGBydW4uanNvbmBgIGZpbGVzIGludG8gbWVhbsKxc3RkIHRhYmxlcyB2cyB0aGUgcGFwZXIuCgpFbWl0cyBgYHJlcG9ydC9jb21wYXJpc29uLmNzdmBgIGFuZCBgYHJlcG9ydC9jb21wYXJpc29uLm1kYGA6IG9uZSByb3cgcGVyCihkYXRhc2V0LCBtb2RlbCwgc2l6ZSkgd2l0aCBvdXIgbWVhbsKxc3RkIEFVQy9BQ0MsIHRoZSBwYXBlcidzIHJlZmVyZW5jZSB2YWx1ZSwKdGhlIHNpZ25lZCBkZWx0YSwgYW5kIGFuIG91dC1vZi10b2xlcmFuY2UgZmxhZy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKaW1wb3J0IGdsb2IKaW1wb3J0IGpzb24KaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC5yZWZlcmVuY2UgaW1wb3J0IFJFRkVSRU5DRQoKIyBUb2xlcmFuY2VzIGZyb20gdGhlIHRhc2sncyBkZWZpbml0aW9uIG9mIGRvbmUuCkFVQ19UT0wgPSAwLjAyCkFDQ19UT0wgPSAwLjAzCgoKZGVmIGxvYWRfcnVucyhyZXN1bHRzX2Rpcj0icmVzdWx0cyIpOgogICAgcnVucyA9IFtdCiAgICBmb3IgcGF0aCBpbiBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihyZXN1bHRzX2RpciwgIioiLCAicnVuLmpzb24iKSkpOgogICAgICAgIHdpdGggb3BlbihwYXRoKSBhcyBmOgogICAgICAgICAgICBydW5zLmFwcGVuZChqc29uLmxvYWQoZikpCiAgICByZXR1cm4gcnVucwoKCmRlZiBfaXNfYmFzZWxpbmUocik6CiAgICBjID0gclsiY29uZmlnIl0KICAgIHJldHVybiAoYy5nZXQoIndpZHRoX211bHQiLCAxLjApID09IDEuMCBhbmQgbm90IGMuZ2V0KCJ3ZWlnaHRlZF9zYW1wbGVyIikKICAgICAgICAgICAgYW5kIG5vdCBjLmdldCgid2VpZ2h0ZWRfbG9zcyIpIGFuZCBub3QgYy5nZXQoInRhZyIpKQoKCmRlZiBhZ2dyZWdhdGUocmVzdWx0c19kaXI9InJlc3VsdHMiLCByZXBvcnRfZGlyPSJyZXBvcnQiKToKICAgIHJ1bnMgPSBbciBmb3IgciBpbiBsb2FkX3J1bnMocmVzdWx0c19kaXIpIGlmIF9pc19iYXNlbGluZShyKV0KICAgIGdyb3VwcyA9IHt9CiAgICBmb3IgciBpbiBydW5zOgogICAgICAgIGMgPSByWyJjb25maWciXQogICAgICAgIGtleSA9IChjWyJkYXRhc2V0Il0sIGNbIm1vZGVsIl0sIGNbInNpemUiXSkKICAgICAgICBncm91cHMuc2V0ZGVmYXVsdChrZXksIFtdKS5hcHBlbmQocikKCiAgICByb3dzID0gW10KICAgIGZvciBrZXkgaW4gc29ydGVkKGdyb3Vwcyk6CiAgICAgICAgZGF0YXNldCwgbW9kZWwsIHNpemUgPSBrZXkKICAgICAgICBnID0gZ3JvdXBzW2tleV0KICAgICAgICBhdWNzID0gbnAuYXJyYXkoW3JbInRlc3RfYXVjIl0gZm9yIHIgaW4gZ10pCiAgICAgICAgYWNjcyA9IG5wLmFycmF5KFtyWyJ0ZXN0X2FjYyJdIGZvciByIGluIGddKQogICAgICAgIHJlZiA9IFJFRkVSRU5DRS5nZXQoa2V5KQogICAgICAgIHJlZl9hdWMsIHJlZl9hY2MgPSAocmVmIGlmIHJlZiBlbHNlIChOb25lLCBOb25lKSkKICAgICAgICBkX2F1YyA9IGZsb2F0KGF1Y3MubWVhbigpIC0gcmVmX2F1YykgaWYgcmVmIGVsc2UgTm9uZQogICAgICAgIGRfYWNjID0gZmxvYXQoYWNjcy5tZWFuKCkgLSByZWZfYWNjKSBpZiByZWYgZWxzZSBOb25lCiAgICAgICAgZmxhZyA9ICIiCiAgICAgICAgaWYgcmVmOgogICAgICAgICAgICBpZiBhYnMoZF9hdWMpID4gQVVDX1RPTCBvciBhYnMoZF9hY2MpID4gQUNDX1RPTDoKICAgICAgICAgICAgICAgIGZsYWcgPSAiT1VUX09GX1RPTCIKICAgICAgICByb3dzLmFwcGVuZChkaWN0KAogICAgICAgICAgICBkYXRhc2V0PWRhdGFzZXQsIG1vZGVsPW1vZGVsLCBzaXplPXNpemUsIG5fc2VlZHM9bGVuKGcpLAogICAgICAgICAgICBhdWNfbWVhbj1mbG9hdChhdWNzLm1lYW4oKSksIGF1Y19zdGQ9ZmxvYXQoYXVjcy5zdGQoZGRvZj0wKSksCiAgICAgICAgICAgIGFjY19tZWFuPWZsb2F0KGFjY3MubWVhbigpKSwgYWNjX3N0ZD1mbG9hdChhY2NzLnN0ZChkZG9mPTApKSwKICAgICAgICAgICAgcmVmX2F1Yz1yZWZfYXVjLCByZWZfYWNjPXJlZl9hY2MsCiAgICAgICAgICAgIGRlbHRhX2F1Yz1kX2F1YywgZGVsdGFfYWNjPWRfYWNjLCBmbGFnPWZsYWcsCiAgICAgICAgKSkKCiAgICBvcy5tYWtlZGlycyhyZXBvcnRfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgX3dyaXRlX2Nzdihyb3dzLCBvcy5wYXRoLmpvaW4ocmVwb3J0X2RpciwgImNvbXBhcmlzb24uY3N2IikpCiAgICBfd3JpdGVfbWQocm93cywgb3MucGF0aC5qb2luKHJlcG9ydF9kaXIsICJjb21wYXJpc29uLm1kIikpCiAgICByZXR1cm4gcm93cwoKCmRlZiBfd3JpdGVfY3N2KHJvd3MsIHBhdGgpOgogICAgaW1wb3J0IGNzdgogICAgZmllbGRzID0gWyJkYXRhc2V0IiwgIm1vZGVsIiwgInNpemUiLCAibl9zZWVkcyIsICJhdWNfbWVhbiIsICJhdWNfc3RkIiwKICAgICAgICAgICAgICAiYWNjX21lYW4iLCAiYWNjX3N0ZCIsICJyZWZfYXVjIiwgInJlZl9hY2MiLCAiZGVsdGFfYXVjIiwKICAgICAgICAgICAgICAiZGVsdGFfYWNjIiwgImZsYWciXQogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgbmV3bGluZT0iIikgYXMgZjoKICAgICAgICB3ID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1maWVsZHMpCiAgICAgICAgdy53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIHIgaW4gcm93czoKICAgICAgICAgICAgdy53cml0ZXJvdyhyKQoKCmRlZiBfZm10KHgsIG5kPTMpOgogICAgcmV0dXJuICIiIGlmIHggaXMgTm9uZSBlbHNlIGYie3g6LntuZH1mfSIKCgpkZWYgX3dyaXRlX21kKHJvd3MsIHBhdGgpOgogICAgbGluZXMgPSBbCiAgICAgICAgIiMgTWVkTU5JU1QgdjIgcmVwbGljYXRpb24g4oCUIGNvbXBhcmlzb24gdnMgcGFwZXIgKFRhYmxlIDMpIiwKICAgICAgICAiIiwKICAgICAgICAiQVVDIC8gQUNDIHJlcG9ydGVkIGFzIG91ciBtZWFuIMKxIHN0ZCBhY3Jvc3Mgc2VlZHM7IGRlbHRhID0gb3VycyDiiJIgcGFwZXIuIiwKICAgICAgICBmIlRvbGVyYW5jZTogfM6UQVVDfCDiiaQge0FVQ19UT0x9LCB8zpRBQ0N8IOKJpCB7QUNDX1RPTH0uIEZsYWcgbWFya3MgY29uZmlncyBvdXRzaWRlIGl0LiIsCiAgICAgICAgIiIsCiAgICAgICAgInwgRGF0YXNldCB8IE1vZGVsIHwgU2l6ZSB8IFNlZWRzIHwgT3VyIEFVQyB8IFBhcGVyIEFVQyB8IM6UQVVDIHwgT3VyIEFDQyB8IFBhcGVyIEFDQyB8IM6UQUNDIHwgRmxhZyB8IiwKICAgICAgICAifC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18IiwKICAgIF0KICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgb3VyX2F1YyA9IGYie3JbJ2F1Y19tZWFuJ106LjNmfSDCsSB7clsnYXVjX3N0ZCddOi4zZn0iCiAgICAgICAgb3VyX2FjYyA9IGYie3JbJ2FjY19tZWFuJ106LjNmfSDCsSB7clsnYWNjX3N0ZCddOi4zZn0iCiAgICAgICAgbGluZXMuYXBwZW5kKAogICAgICAgICAgICBmInwge3JbJ2RhdGFzZXQnXX0gfCB7clsnbW9kZWwnXX0gfCB7clsnc2l6ZSddfSB8IHtyWyduX3NlZWRzJ119IHwgIgogICAgICAgICAgICBmIntvdXJfYXVjfSB8IHtfZm10KHJbJ3JlZl9hdWMnXSl9IHwge19mbXQoclsnZGVsdGFfYXVjJ10pfSB8ICIKICAgICAgICAgICAgZiJ7b3VyX2FjY30gfCB7X2ZtdChyWydyZWZfYWNjJ10pfSB8IHtfZm10KHJbJ2RlbHRhX2FjYyddKX0gfCAiCiAgICAgICAgICAgIGYie3JbJ2ZsYWcnXX0gfCIpCiAgICBsaW5lcy5hcHBlbmQoIiIpCiAgICB3aXRoIG9wZW4ocGF0aCwgInciKSBhcyBmOgogICAgICAgIGYud3JpdGUoIlxuIi5qb2luKGxpbmVzKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaW1wb3J0IHN5cwogICAgcmQgPSBzeXMuYXJndlsxXSBpZiBsZW4oc3lzLmFyZ3YpID4gMSBlbHNlICJyZXN1bHRzIgogICAgcm93cyA9IGFnZ3JlZ2F0ZShyZCkKICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgcHJpbnQocikK',
    'data.py': 'IiIiRGF0YXNldCBsb2FkaW5nIGFuZCB0cmFuc2Zvcm0gcGlwZWxpbmVzLgoKV2UgdGFrZSBvbmx5IHRoZSBzdGFuZGFyZGl6ZWQgZGF0YXNldCBmcm9tIHRoZSBgYG1lZG1uaXN0YGAgcGFja2FnZSAoaXRzIGxvYWRlcgpjbGFzc2VzKSBwbHVzIHRoZSBsYWJlbCBhcnJheS4gRXZlcnl0aGluZyBlbHNlIGhlcmUgaXMgb3Vycy4KCktleSBwcm90b2NvbCBwb2ludHMsIGZhaXRoZnVsIHRvIE1lZE1OSVNUIHYyICgyMDIzKToKCiogQWx3YXlzIGxvYWQgdGhlICoqMjgtcGl4ZWwqKiBgYC5ucHpgYCAoYGBzaXplPTI4YGApLCBgYGFzX3JnYj1UcnVlYGAuCiogVGhlIDIyNCBjb25maWdzICpyZXNpemUgdGhlIDI4LXBpeGVsIGRhdGEgaW5zaWRlIHRoZSB0cmFuc2Zvcm0qIHVzaW5nCiAgbmVhcmVzdC1uZWlnaGJvdXIgaW50ZXJwb2xhdGlvbi4gV2UgZGVsaWJlcmF0ZWx5IGRvICoqbm90KiogbG9hZCBNZWRNTklTVCsncwogIG5hdGl2ZSAyMjQgaW1hZ2VzLCB3aGljaCBkaWQgbm90IGV4aXN0IGZvciB0aGUgMjAyMyBwYXBlciBhbmQgd291bGQgZ2l2ZQogIGRpZmZlcmVudCAoYmV0dGVyKSBudW1iZXJzLgoqIE5vcm1hbGlzYXRpb24gaXMgYGBtZWFuPVsuNV0sIHN0ZD1bLjVdYGAgYnJvYWRjYXN0IGFjcm9zcyB0aGUgMyBjaGFubmVscwogIChyb3VnaGx5IG1hcHMgdG8gWy0xLCAxXSkuIE5vIHRyYWluLXRpbWUgYXVnbWVudGF0aW9uIGluIHRoZSBiYXNlbGluZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaHZpc2lvbi50cmFuc2Zvcm1zIGFzIFQKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgppbXBvcnQgbWVkbW5pc3QKZnJvbSBtZWRtbmlzdCBpbXBvcnQgSU5GTwoKCmRlZiBidWlsZF90cmFuc2Zvcm0oc2l6ZSk6CiAgICAiIiJUcmFuc2Zvcm0gZm9yIHRoZSBnaXZlbiByZXNvbHV0aW9uIChubyBhdWdtZW50YXRpb247IGJhc2VsaW5lKS4iIiIKICAgIGlmIHNpemUgPT0gMjg6CiAgICAgICAgcmV0dXJuIFQuQ29tcG9zZShbCiAgICAgICAgICAgIFQuVG9UZW5zb3IoKSwKICAgICAgICAgICAgVC5Ob3JtYWxpemUobWVhbj1bLjVdLCBzdGQ9Wy41XSksCiAgICAgICAgXSkKICAgIGlmIHNpemUgPT0gMjI0OgogICAgICAgICMgTmVhcmVzdC1uZWlnaGJvdXIgc3BlY2lmaWNhbGx5OyB0aGUgMjgtcGl4ZWwgc291cmNlIGlzIHVwc2FtcGxlZCBoZXJlLgogICAgICAgIHJldHVybiBULkNvbXBvc2UoWwogICAgICAgICAgICBULlJlc2l6ZSgoMjI0LCAyMjQpLCBpbnRlcnBvbGF0aW9uPUltYWdlLk5FQVJFU1QpLAogICAgICAgICAgICBULlRvVGVuc29yKCksCiAgICAgICAgICAgIFQuTm9ybWFsaXplKG1lYW49Wy41XSwgc3RkPVsuNV0pLAogICAgICAgIF0pCiAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5zdXBwb3J0ZWQgc2l6ZSB7c2l6ZX0iKQoKCmRlZiBnZXRfaW5mbyhkYXRhc2V0KToKICAgIHJldHVybiBJTkZPW2RhdGFzZXRdCgoKZGVmIGdldF9kYXRhc2V0KGRhdGFzZXQsIHNwbGl0LCBzaXplLCByb290PU5vbmUsIGRvd25sb2FkPVRydWUpOgogICAgIiIiUmV0dXJuIGEgYGBtZWRtbmlzdGBgIGRhdGFzZXQgb2JqZWN0IGZvciBgYHNwbGl0YGAgd2l0aCBvdXIgdHJhbnNmb3JtLgoKICAgIERhdGEgaXMgYWx3YXlzIGxvYWRlZCBmcm9tIHRoZSAyOC1waXhlbCBgYC5ucHpgYDsgYGBzaXplYGAgb25seSBjb250cm9scyB0aGUKICAgIHRyYW5zZm9ybSAodGhlIDIyNCBwaXBlbGluZSByZXNpemVzIGludGVybmFsbHkpLgogICAgIiIiCiAgICBpbmZvID0gSU5GT1tkYXRhc2V0XQogICAgRGF0YUNsYXNzID0gZ2V0YXR0cihtZWRtbmlzdCwgaW5mb1sicHl0aG9uX2NsYXNzIl0pCiAgICB0cmFuc2Zvcm0gPSBidWlsZF90cmFuc2Zvcm0oc2l6ZSkKICAgIGt3YXJncyA9IGRpY3Qoc3BsaXQ9c3BsaXQsIHRyYW5zZm9ybT10cmFuc2Zvcm0sIGRvd25sb2FkPWRvd25sb2FkLAogICAgICAgICAgICAgICAgICBhc19yZ2I9VHJ1ZSwgc2l6ZT0yOCkKICAgIGlmIHJvb3QgaXMgbm90IE5vbmU6CiAgICAgICAga3dhcmdzWyJyb290Il0gPSByb290CiAgICByZXR1cm4gRGF0YUNsYXNzKCoqa3dhcmdzKQoKCmRlZiBnZXRfbG9hZGVycyhkYXRhc2V0LCBzaXplLCBiYXRjaF9zaXplPTEyOCwgcm9vdD1Ob25lLCBkb3dubG9hZD1UcnVlLAogICAgICAgICAgICAgICAgbnVtX3dvcmtlcnM9Miwgc2FtcGxlcj1Ob25lLCBldmFsX2JhdGNoX3NpemU9Tm9uZSwKICAgICAgICAgICAgICAgIHBpbl9tZW1vcnk9VHJ1ZSk6CiAgICAiIiJCdWlsZCB0cmFpbi92YWwvdGVzdCBkYXRhbG9hZGVycy4KCiAgICBgYHNhbXBsZXJgYCAoZS5nLiBhIGBgV2VpZ2h0ZWRSYW5kb21TYW1wbGVyYGApIHJlcGxhY2VzIHNodWZmbGluZyBvbiB0aGUKICAgIHRyYWluIGxvYWRlciB3aGVuIHByb3ZpZGVkOyB1c2VkIGJ5IHRoZSBiaWFzLW1pdGlnYXRpb24gZXh0ZW5zaW9uLgogICAgIiIiCiAgICB0cmFpbl9zZXQgPSBnZXRfZGF0YXNldChkYXRhc2V0LCAidHJhaW4iLCBzaXplLCByb290LCBkb3dubG9hZCkKICAgIHZhbF9zZXQgPSBnZXRfZGF0YXNldChkYXRhc2V0LCAidmFsIiwgc2l6ZSwgcm9vdCwgZG93bmxvYWQpCiAgICB0ZXN0X3NldCA9IGdldF9kYXRhc2V0KGRhdGFzZXQsICJ0ZXN0Iiwgc2l6ZSwgcm9vdCwgZG93bmxvYWQpCgogICAgZXZhbF9icyA9IGV2YWxfYmF0Y2hfc2l6ZSBvciBiYXRjaF9zaXplCiAgICBzaHVmZmxlID0gc2FtcGxlciBpcyBOb25lCiAgICB0cmFpbl9sb2FkZXIgPSB0b3JjaC51dGlscy5kYXRhLkRhdGFMb2FkZXIoCiAgICAgICAgdHJhaW5fc2V0LCBiYXRjaF9zaXplPWJhdGNoX3NpemUsIHNodWZmbGU9c2h1ZmZsZSwgc2FtcGxlcj1zYW1wbGVyLAogICAgICAgIG51bV93b3JrZXJzPW51bV93b3JrZXJzLCBwaW5fbWVtb3J5PXBpbl9tZW1vcnksIGRyb3BfbGFzdD1GYWxzZSkKICAgIHZhbF9sb2FkZXIgPSB0b3JjaC51dGlscy5kYXRhLkRhdGFMb2FkZXIoCiAgICAgICAgdmFsX3NldCwgYmF0Y2hfc2l6ZT1ldmFsX2JzLCBzaHVmZmxlPUZhbHNlLAogICAgICAgIG51bV93b3JrZXJzPW51bV93b3JrZXJzLCBwaW5fbWVtb3J5PXBpbl9tZW1vcnkpCiAgICB0ZXN0X2xvYWRlciA9IHRvcmNoLnV0aWxzLmRhdGEuRGF0YUxvYWRlcigKICAgICAgICB0ZXN0X3NldCwgYmF0Y2hfc2l6ZT1ldmFsX2JzLCBzaHVmZmxlPUZhbHNlLAogICAgICAgIG51bV93b3JrZXJzPW51bV93b3JrZXJzLCBwaW5fbWVtb3J5PXBpbl9tZW1vcnkpCiAgICByZXR1cm4gdHJhaW5fbG9hZGVyLCB2YWxfbG9hZGVyLCB0ZXN0X2xvYWRlcgoKCmRlZiBjbGFzc19jb3VudHMoZGF0YXNldCwgcm9vdD1Ob25lLCBkb3dubG9hZD1UcnVlKToKICAgICIiIlBlci1jbGFzcyB0cmFpbmluZy1zYW1wbGUgY291bnRzIChtdWx0aS1jbGFzcyBkYXRhc2V0cykuIiIiCiAgICB0cmFpbl9zZXQgPSBnZXRfZGF0YXNldChkYXRhc2V0LCAidHJhaW4iLCAyOCwgcm9vdCwgZG93bmxvYWQpCiAgICBsYWJlbHMgPSBucC5hc2FycmF5KHRyYWluX3NldC5sYWJlbHMpLnNxdWVlemUoKS5hc3R5cGUoaW50KQogICAgbl9jbGFzc2VzID0gbGVuKElORk9bZGF0YXNldF1bImxhYmVsIl0pCiAgICBjb3VudHMgPSBucC5iaW5jb3VudChsYWJlbHMsIG1pbmxlbmd0aD1uX2NsYXNzZXMpCiAgICByZXR1cm4gY291bnRzCgoKZGVmIGNsYXNzX3dlaWdodHMoZGF0YXNldCwgcm9vdD1Ob25lLCBkb3dubG9hZD1UcnVlLCBub3JtYWxpemU9VHJ1ZSk6CiAgICAiIiJJbnZlcnNlLWZyZXF1ZW5jeSBjbGFzcyB3ZWlnaHRzIGZvciB3ZWlnaHRlZCBDcm9zc0VudHJvcHlMb3NzLgoKICAgIGBgd19jIOKInSAxIC8gY291bnRfY2BgOyBvcHRpb25hbGx5IG5vcm1hbGlzZWQgdG8gbWVhbiAxLgogICAgIiIiCiAgICBjb3VudHMgPSBjbGFzc19jb3VudHMoZGF0YXNldCwgcm9vdCwgZG93bmxvYWQpLmFzdHlwZShucC5mbG9hdDY0KQogICAgY291bnRzID0gbnAuY2xpcChjb3VudHMsIDEsIE5vbmUpCiAgICB3ID0gMS4wIC8gY291bnRzCiAgICBpZiBub3JtYWxpemU6CiAgICAgICAgdyA9IHcgKiBsZW4odykgLyB3LnN1bSgpCiAgICByZXR1cm4gdG9yY2gudGVuc29yKHcsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCgoKZGVmIG1ha2Vfd2VpZ2h0ZWRfc2FtcGxlcihkYXRhc2V0LCByb290PU5vbmUsIGRvd25sb2FkPVRydWUpOgogICAgIiIiQSBgYFdlaWdodGVkUmFuZG9tU2FtcGxlcmBgIGdpdmluZyBlYWNoIGNsYXNzIGVxdWFsIGV4cGVjdGVkIG1hc3MuIiIiCiAgICB0cmFpbl9zZXQgPSBnZXRfZGF0YXNldChkYXRhc2V0LCAidHJhaW4iLCAyOCwgcm9vdCwgZG93bmxvYWQpCiAgICBsYWJlbHMgPSBucC5hc2FycmF5KHRyYWluX3NldC5sYWJlbHMpLnNxdWVlemUoKS5hc3R5cGUoaW50KQogICAgY291bnRzID0gbnAuYmluY291bnQobGFiZWxzLCBtaW5sZW5ndGg9bGVuKElORk9bZGF0YXNldF1bImxhYmVsIl0pKS5hc3R5cGUobnAuZmxvYXQ2NCkKICAgIGNvdW50cyA9IG5wLmNsaXAoY291bnRzLCAxLCBOb25lKQogICAgc2FtcGxlX3cgPSAoMS4wIC8gY291bnRzKVtsYWJlbHNdCiAgICBzYW1wbGVyID0gdG9yY2gudXRpbHMuZGF0YS5XZWlnaHRlZFJhbmRvbVNhbXBsZXIoCiAgICAgICAgd2VpZ2h0cz10b3JjaC50ZW5zb3Ioc2FtcGxlX3csIGR0eXBlPXRvcmNoLmRvdWJsZSksCiAgICAgICAgbnVtX3NhbXBsZXM9bGVuKHNhbXBsZV93KSwgcmVwbGFjZW1lbnQ9VHJ1ZSkKICAgIHJldHVybiBzYW1wbGVyCg==',
    'evaluate.py': 'IiIiSW5mZXJlbmNlLCBwcmVkaWN0aW9uIGR1bXBpbmcsIGFuZCBjaGVja3BvaW50IGV2YWx1YXRpb24uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCmZyb20gLiBpbXBvcnQgbWV0cmljcwoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIHByZWRpY3QobW9kZWwsIGxvYWRlciwgdGFzaywgZGV2aWNlLCB1c2VfYW1wPUZhbHNlKToKICAgICIiIlJ1biB0aGUgbW9kZWwgb3ZlciBgYGxvYWRlcmBgOyByZXR1cm4gYGAoeV90cnVlLCB5X3Njb3JlKWBgIG51bXB5IGFycmF5cy4KCiAgICBgYHlfc2NvcmVgYCBpcyBzb2Z0bWF4IHByb2JhYmlsaXRpZXMgKG11bHRpLWNsYXNzKSBvciBzaWdtb2lkcwogICAgKG11bHRpLWxhYmVsKS4gYGB5X3RydWVgYCBpcyBgYChOLCAxKWBgIGludCBmb3IgbXVsdGktY2xhc3MsIGBgKE4sIEMpYGAgZm9yCiAgICBtdWx0aS1sYWJlbCAtLSBtYXRjaGluZyB3aGF0IGBgbWVkbW5pc3QuRXZhbHVhdG9yYGAgZXhwZWN0cy4KICAgICIiIgogICAgbW9kZWwuZXZhbCgpCiAgICBzY29yZXMsIHRydWVzID0gW10sIFtdCiAgICBmb3IgeCwgeSBpbiBsb2FkZXI6CiAgICAgICAgeCA9IHgudG8oZGV2aWNlLCBub25fYmxvY2tpbmc9VHJ1ZSkKICAgICAgICB3aXRoIHRvcmNoLmN1ZGEuYW1wLmF1dG9jYXN0KGVuYWJsZWQ9dXNlX2FtcCk6CiAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKHgpCiAgICAgICAgbG9naXRzID0gbG9naXRzLmZsb2F0KCkKICAgICAgICBpZiB0YXNrID09ICJtdWx0aS1sYWJlbCwgYmluYXJ5LWNsYXNzIjoKICAgICAgICAgICAgcyA9IHRvcmNoLnNpZ21vaWQobG9naXRzKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHMgPSBGLnNvZnRtYXgobG9naXRzLCBkaW09MSkKICAgICAgICBzY29yZXMuYXBwZW5kKHMuY3B1KCkubnVtcHkoKSkKICAgICAgICB0cnVlcy5hcHBlbmQoeS5udW1weSgpKQogICAgeV9zY29yZSA9IG5wLmNvbmNhdGVuYXRlKHNjb3JlcywgYXhpcz0wKQogICAgeV90cnVlID0gbnAuY29uY2F0ZW5hdGUodHJ1ZXMsIGF4aXM9MCkKICAgIHJldHVybiB5X3RydWUsIHlfc2NvcmUKCgpkZWYgZXZhbHVhdGVfc3BsaXQobW9kZWwsIGxvYWRlciwgdGFzaywgZGV2aWNlLCB1c2VfYW1wPUZhbHNlKToKICAgICIiIlJldHVybiBgYChhdWMsIGFjYywgeV90cnVlLCB5X3Njb3JlKWBgIGZvciBhIHNwbGl0LiIiIgogICAgeV90cnVlLCB5X3Njb3JlID0gcHJlZGljdChtb2RlbCwgbG9hZGVyLCB0YXNrLCBkZXZpY2UsIHVzZV9hbXApCiAgICBhdWMsIGFjYyA9IG1ldHJpY3MuZXZhbHVhdGUoeV90cnVlLCB5X3Njb3JlLCB0YXNrKQogICAgcmV0dXJuIGF1YywgYWNjLCB5X3RydWUsIHlfc2NvcmUKCgpkZWYgcHJlZGljdGlvbl9maWxlbmFtZShmbGFnLCBzcGxpdCwgYXVjLCBhY2MsIHNlZWQsIHNpemU9MjgpOgogICAgIiIiTWVkTU5JU1QtY29udmVudGlvbiBzZWxmLWRlc2NyaWJpbmcgZmlsZW5hbWUuIiIiCiAgICBzaXplX2ZsYWcgPSAiIiBpZiBzaXplID09IDI4IGVsc2UgZiJfe3NpemV9IgogICAgIyBzaXplX2ZsYWcgaXMgZm9sZGVkIGludG8gYGZsYWdgIHBvc2l0aW9uIHBlciBtZWRtbmlzdCBwYXJzZXIgZXhwZWN0YXRpb25zLgogICAgcmV0dXJuIGYie2ZsYWd9e3NpemVfZmxhZ31fe3NwbGl0fV9bQVVDXXthdWM6LjNmfV9bQUNDXXthY2M6LjNmfUBzZWVke3NlZWR9LmNzdiIKCgpkZWYgc2F2ZV9wcmVkaWN0aW9ucyh5X3Njb3JlLCBwYXRoKToKICAgICIiIlNhdmUgc2NvcmVzIGluIHRoZSBtZWRtbmlzdCByZXN1bHQgZm9ybWF0IChpbmRleCwgc2NvcmVfMCwgLi4uKS4iIiIKICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShwYXRoKSBvciAiLiIsIGV4aXN0X29rPVRydWUpCiAgICBwZC5EYXRhRnJhbWUoeV9zY29yZSkudG9fY3N2KHBhdGgsIGhlYWRlcj1Ob25lKQoKCmRlZiBsb2FkX2NoZWNrcG9pbnQocGF0aCwgbW9kZWwsIG9wdGltaXplcj1Ob25lLCBzY2hlZHVsZXI9Tm9uZSwgc2NhbGVyPU5vbmUsCiAgICAgICAgICAgICAgICAgICAgbWFwX2xvY2F0aW9uPSJjcHUiKToKICAgIGNrcHQgPSB0b3JjaC5sb2FkKHBhdGgsIG1hcF9sb2NhdGlvbj1tYXBfbG9jYXRpb24pCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2twdFsibW9kZWwiXSkKICAgIGlmIG9wdGltaXplciBpcyBub3QgTm9uZSBhbmQgY2twdC5nZXQoIm9wdGltaXplciIpIGlzIG5vdCBOb25lOgogICAgICAgIG9wdGltaXplci5sb2FkX3N0YXRlX2RpY3QoY2twdFsib3B0aW1pemVyIl0pCiAgICBpZiBzY2hlZHVsZXIgaXMgbm90IE5vbmUgYW5kIGNrcHQuZ2V0KCJzY2hlZHVsZXIiKSBpcyBub3QgTm9uZToKICAgICAgICBzY2hlZHVsZXIubG9hZF9zdGF0ZV9kaWN0KGNrcHRbInNjaGVkdWxlciJdKQogICAgaWYgc2NhbGVyIGlzIG5vdCBOb25lIGFuZCBja3B0LmdldCgic2NhbGVyIikgaXMgbm90IE5vbmU6CiAgICAgICAgc2NhbGVyLmxvYWRfc3RhdGVfZGljdChja3B0WyJzY2FsZXIiXSkKICAgIHJldHVybiBja3B0Cg==',
    'extensions.py': '...',
    'figures_replication.py': '...',
    'metrics.py': 'IiIiT3VyIG93biBpbXBsZW1lbnRhdGlvbiBvZiB0aGUgTWVkTU5JU1QgZXZhbHVhdGlvbiBtZXRyaWNzLgoKV2UgcmVpbXBsZW1lbnQgbWFjcm8gb25lLXZzLXJlc3QgUk9DLUFVQyBhbmQgYXJnbWF4IGFjY3VyYWN5IGZyb20gc2NyYXRjaCBhbmQKcHJvdmlkZSA6ZnVuYzpgY2hlY2tfYWdyZWVtZW50YCwgd2hpY2ggYXNzZXJ0cyBvdXIgbnVtYmVycyBtYXRjaApgYG1lZG1uaXN0LkV2YWx1YXRvcmBgICh1c2VkICpvbmx5KiBhcyBhbiBvcmFjbGUpIHRvIHdpdGhpbiBhIHRvbGVyYW5jZS4KCk5vdGhpbmcgaGVyZSBpcyBjb3BpZWQgZnJvbSBgYE1lZE1OSVNUL2V4cGVyaW1lbnRzYGA7IHRoZSBvbmx5IE1lZE1OSVNUIGNvZGUgd2UKdG91Y2ggaXMgdGhlIGBgbWVkbW5pc3RgYCBQeVBJIHBhY2thZ2UncyBgYEV2YWx1YXRvcmBgIGluIHRoZSB2ZXJpZmljYXRpb24gcGF0aC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJvY19hdWNfc2NvcmUsIGFjY3VyYWN5X3Njb3JlCgoKZGVmIF9zcXVlZXplKHlfdHJ1ZSwgeV9zY29yZSk6CiAgICByZXR1cm4gbnAuYXNhcnJheSh5X3RydWUpLnNxdWVlemUoKSwgbnAuYXNhcnJheSh5X3Njb3JlKS5zcXVlZXplKCkKCgpkZWYgZ2V0QVVDKHlfdHJ1ZSwgeV9zY29yZSwgdGFzayk6CiAgICAiIiJNYWNyby1hdmVyYWdlZCBBVUMuCgogICAgRm9yIG11bHRpLWNsYXNzIHRhc2tzIHRoaXMgaXMgdGhlIG1lYW4gb3ZlciBjbGFzc2VzIG9mIHRoZSBvbmUtdnMtcmVzdAogICAgUk9DLUFVQyAoYGAoeV90cnVlID09IGMpYGAgdnMgYGB5X3Njb3JlWzosIGNdYGApLCB3aGljaCBpcyBleGFjdGx5IHdoYXQKICAgIHNjaWtpdC1sZWFybidzIGBgcm9jX2F1Y19zY29yZShhdmVyYWdlPSdtYWNybycsIG11bHRpX2NsYXNzPSdvdnInKWBgIGFuZAogICAgYGBtZWRtbmlzdC5FdmFsdWF0b3JgYCBjb21wdXRlLiBgYG11bHRpLWxhYmVsLCBiaW5hcnktY2xhc3NgYCBhdmVyYWdlcyB0aGUKICAgIHBlci1sYWJlbCBiaW5hcnkgQVVDOyBgYGJpbmFyeS1jbGFzc2BgIHVzZXMgdGhlIHBvc2l0aXZlLWNsYXNzIHNjb3JlLgogICAgIiIiCiAgICB5X3RydWUsIHlfc2NvcmUgPSBfc3F1ZWV6ZSh5X3RydWUsIHlfc2NvcmUpCgogICAgaWYgdGFzayA9PSAibXVsdGktbGFiZWwsIGJpbmFyeS1jbGFzcyI6CiAgICAgICAgYXVjcyA9IFtyb2NfYXVjX3Njb3JlKHlfdHJ1ZVs6LCBpXSwgeV9zY29yZVs6LCBpXSkgZm9yIGkgaW4gcmFuZ2UoeV9zY29yZS5zaGFwZVsxXSldCiAgICAgICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4oYXVjcykpCgogICAgaWYgdGFzayA9PSAiYmluYXJ5LWNsYXNzIjoKICAgICAgICBpZiB5X3Njb3JlLm5kaW0gPT0gMjoKICAgICAgICAgICAgeV9zY29yZSA9IHlfc2NvcmVbOiwgLTFdCiAgICAgICAgcmV0dXJuIGZsb2F0KHJvY19hdWNfc2NvcmUoeV90cnVlLCB5X3Njb3JlKSkKCiAgICAjIG11bHRpLWNsYXNzIC8gb3JkaW5hbC1yZWdyZXNzaW9uOiBtYWNybyBvbmUtdnMtcmVzdC4KICAgIGF1Y3MgPSBbXQogICAgZm9yIGMgaW4gcmFuZ2UoeV9zY29yZS5zaGFwZVsxXSk6CiAgICAgICAgYXVjcy5hcHBlbmQocm9jX2F1Y19zY29yZSgoeV90cnVlID09IGMpLmFzdHlwZShmbG9hdCksIHlfc2NvcmVbOiwgY10pKQogICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4oYXVjcykpCgoKZGVmIGdldEFDQyh5X3RydWUsIHlfc2NvcmUsIHRhc2ssIHRocmVzaG9sZD0wLjUpOgogICAgIiIiQWNjdXJhY3kuIEFyZ21heCBmb3IgbXVsdGktY2xhc3M7IHBlci1sYWJlbCB0aHJlc2hvbGRpbmcgb3RoZXJ3aXNlLiIiIgogICAgeV90cnVlLCB5X3Njb3JlID0gX3NxdWVlemUoeV90cnVlLCB5X3Njb3JlKQoKICAgIGlmIHRhc2sgPT0gIm11bHRpLWxhYmVsLCBiaW5hcnktY2xhc3MiOgogICAgICAgIHlfcHJlZCA9IHlfc2NvcmUgPiB0aHJlc2hvbGQKICAgICAgICBhY2NzID0gW2FjY3VyYWN5X3Njb3JlKHlfdHJ1ZVs6LCBpXSwgeV9wcmVkWzosIGldKSBmb3IgaSBpbiByYW5nZSh5X3RydWUuc2hhcGVbMV0pXQogICAgICAgIHJldHVybiBmbG9hdChucC5tZWFuKGFjY3MpKQoKICAgIGlmIHRhc2sgPT0gImJpbmFyeS1jbGFzcyI6CiAgICAgICAgaWYgeV9zY29yZS5uZGltID09IDI6CiAgICAgICAgICAgIHlfc2NvcmUgPSB5X3Njb3JlWzosIC0xXQogICAgICAgIHJldHVybiBmbG9hdChhY2N1cmFjeV9zY29yZSh5X3RydWUsIHlfc2NvcmUgPiB0aHJlc2hvbGQpKQoKICAgIHJldHVybiBmbG9hdChhY2N1cmFjeV9zY29yZSh5X3RydWUsIG5wLmFyZ21heCh5X3Njb3JlLCBheGlzPS0xKSkpCgoKZGVmIGV2YWx1YXRlKHlfdHJ1ZSwgeV9zY29yZSwgdGFzayk6CiAgICAiIiJSZXR1cm4gYGAoYXVjLCBhY2MpYGAgZm9yIGEgc2V0IG9mIHByZWRpY3Rpb25zLiIiIgogICAgcmV0dXJuIGdldEFVQyh5X3RydWUsIHlfc2NvcmUsIHRhc2spLCBnZXRBQ0MoeV90cnVlLCB5X3Njb3JlLCB0YXNrKQoKCmRlZiBwZXJfY2xhc3NfbWV0cmljcyh5X3RydWUsIHlfc2NvcmUsIG51bV9jbGFzc2VzKToKICAgICIiIlBlci1jbGFzcyBBVUMgLyBwcmVjaXNpb24gLyByZWNhbGwgLyBGMSAobXVsdGktY2xhc3Mgb25seSkuCgogICAgUmV0dXJucyBhIGxpc3Qgb2YgZGljdHMsIG9uZSBwZXIgY2xhc3MuIFVzZWQgYnkgdGhlIHBlci1jbGFzcyBleHRlbnNpb24uCiAgICAiIiIKICAgIGZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBwcmVjaXNpb25fcmVjYWxsX2ZzY29yZV9zdXBwb3J0LCBhdmVyYWdlX3ByZWNpc2lvbl9zY29yZQoKICAgIHlfdHJ1ZSwgeV9zY29yZSA9IF9zcXVlZXplKHlfdHJ1ZSwgeV9zY29yZSkKICAgIHlfcHJlZCA9IG5wLmFyZ21heCh5X3Njb3JlLCBheGlzPS0xKQogICAgcHJlY2lzaW9uLCByZWNhbGwsIGYxLCBzdXBwb3J0ID0gcHJlY2lzaW9uX3JlY2FsbF9mc2NvcmVfc3VwcG9ydCgKICAgICAgICB5X3RydWUsIHlfcHJlZCwgbGFiZWxzPWxpc3QocmFuZ2UobnVtX2NsYXNzZXMpKSwgemVyb19kaXZpc2lvbj0wCiAgICApCiAgICByb3dzID0gW10KICAgIGZvciBjIGluIHJhbmdlKG51bV9jbGFzc2VzKToKICAgICAgICB5X2JpbiA9ICh5X3RydWUgPT0gYykuYXN0eXBlKGZsb2F0KQogICAgICAgIHRyeToKICAgICAgICAgICAgYXVjX2MgPSByb2NfYXVjX3Njb3JlKHlfYmluLCB5X3Njb3JlWzosIGNdKQogICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOiAgIyBhIGNsYXNzIGFic2VudCBmcm9tIHRoaXMgc3BsaXQKICAgICAgICAgICAgYXVjX2MgPSBmbG9hdCgibmFuIikKICAgICAgICB0cnk6CiAgICAgICAgICAgICMgUFItQVVDIChhdmVyYWdlIHByZWNpc2lvbik6IHRoZSBob25lc3QgcmVhZCB1bmRlciBpbWJhbGFuY2UsIHdoZXJlCiAgICAgICAgICAgICMgUk9DLUFVQyBpcyBvcHRpbWlzdGljIGZvciB0aGUgcmFyZSBjbGFzc2VzLgogICAgICAgICAgICBhcF9jID0gYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUoeV9iaW4sIHlfc2NvcmVbOiwgY10pCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIGFwX2MgPSBmbG9hdCgibmFuIikKICAgICAgICByb3dzLmFwcGVuZCgKICAgICAgICAgICAgZGljdCgKICAgICAgICAgICAgICAgIGNscz1pbnQoYyksCiAgICAgICAgICAgICAgICBhdWM9ZmxvYXQoYXVjX2MpLAogICAgICAgICAgICAgICAgYXA9ZmxvYXQoYXBfYyksCiAgICAgICAgICAgICAgICBwcmVjaXNpb249ZmxvYXQocHJlY2lzaW9uW2NdKSwKICAgICAgICAgICAgICAgIHJlY2FsbD1mbG9hdChyZWNhbGxbY10pLAogICAgICAgICAgICAgICAgZjE9ZmxvYXQoZjFbY10pLAogICAgICAgICAgICAgICAgc3VwcG9ydD1pbnQoc3VwcG9ydFtjXSksCiAgICAgICAgICAgICkKICAgICAgICApCiAgICByZXR1cm4gcm93cwoKCmRlZiBjaGVja19hZ3JlZW1lbnQoeV90cnVlLCB5X3Njb3JlLCB0YXNrLCBmbGFnPU5vbmUsIHNwbGl0PU5vbmUsIHNpemU9MjgsCiAgICAgICAgICAgICAgICAgICAgcm9vdD1Ob25lLCB0b2w9MWUtMywgdmVyYm9zZT1UcnVlKToKICAgICIiIkFzc2VydCBvdXIgbWV0cmljcyBhZ3JlZSB3aXRoIGBgbWVkbW5pc3QuRXZhbHVhdG9yYGAgd2l0aGluIGBgdG9sYGAuCgogICAgVHdvIGluZGVwZW5kZW50IG9yYWNsZSBwYXRocyBhcmUgY2hlY2tlZDoKCiAgICAxLiBgYG1lZG1uaXN0LmV2YWx1YXRvci5nZXRBVUMvZ2V0QUNDYGAgb24gdGhlIHNhbWUgYXJyYXlzLgogICAgMi4gSWYgYGBmbGFnYGAvYGBzcGxpdGBgIGFyZSBnaXZlbiwgYSByZWFsIGBgbWVkbW5pc3QuRXZhbHVhdG9yYGAgb2JqZWN0CiAgICAgICAod2hpY2ggcmVsb2FkcyBsYWJlbHMgZnJvbSB0aGUgYGAubnB6YGAgb24gZGlzaykgZXZhbHVhdGluZyBgYHlfc2NvcmVgYC4KCiAgICBSYWlzZXMgYGBBc3NlcnRpb25FcnJvcmBgIGlmIGVpdGhlciBkcmlmdHMgYmV5b25kIGBgdG9sYGAuCiAgICAiIiIKICAgIGZyb20gbWVkbW5pc3QuZXZhbHVhdG9yIGltcG9ydCBnZXRBVUMgYXMgcmVmX2dldEFVQywgZ2V0QUNDIGFzIHJlZl9nZXRBQ0MKCiAgICBvdXJfYXVjID0gZ2V0QVVDKHlfdHJ1ZSwgeV9zY29yZSwgdGFzaykKICAgIG91cl9hY2MgPSBnZXRBQ0MoeV90cnVlLCB5X3Njb3JlLCB0YXNrKQoKICAgIHJlZl9hdWMgPSByZWZfZ2V0QVVDKG5wLmFzYXJyYXkoeV90cnVlKSwgbnAuYXNhcnJheSh5X3Njb3JlKSwgdGFzaykKICAgIHJlZl9hY2MgPSByZWZfZ2V0QUNDKG5wLmFzYXJyYXkoeV90cnVlKSwgbnAuYXNhcnJheSh5X3Njb3JlKSwgdGFzaykKCiAgICBhc3NlcnQgYWJzKG91cl9hdWMgLSByZWZfYXVjKSA8IHRvbCwgZiJBVUMgZHJpZnQ6IG91cnM9e291cl9hdWM6LjZmfSByZWY9e3JlZl9hdWM6LjZmfSIKICAgIGFzc2VydCBhYnMob3VyX2FjYyAtIHJlZl9hY2MpIDwgdG9sLCBmIkFDQyBkcmlmdDogb3Vycz17b3VyX2FjYzouNmZ9IHJlZj17cmVmX2FjYzouNmZ9IgoKICAgIGV2YWxfYXVjID0gZXZhbF9hY2MgPSBOb25lCiAgICBpZiBmbGFnIGlzIG5vdCBOb25lIGFuZCBzcGxpdCBpcyBub3QgTm9uZToKICAgICAgICBmcm9tIG1lZG1uaXN0IGltcG9ydCBFdmFsdWF0b3IKCiAgICAgICAga3dhcmdzID0ge30gaWYgcm9vdCBpcyBOb25lIGVsc2UgeyJyb290Ijogcm9vdH0KICAgICAgICBldmFsdWF0b3IgPSBFdmFsdWF0b3IoZmxhZywgc3BsaXQsIHNpemU9c2l6ZSwgKiprd2FyZ3MpCiAgICAgICAgbSA9IGV2YWx1YXRvci5ldmFsdWF0ZShucC5hc2FycmF5KHlfc2NvcmUpKQogICAgICAgIGV2YWxfYXVjLCBldmFsX2FjYyA9IGZsb2F0KG0uQVVDKSwgZmxvYXQobS5BQ0MpCiAgICAgICAgYXNzZXJ0IGFicyhvdXJfYXVjIC0gZXZhbF9hdWMpIDwgdG9sLCBmIkFVQyBkcmlmdCB2cyBFdmFsdWF0b3I6IG91cnM9e291cl9hdWM6LjZmfSByZWY9e2V2YWxfYXVjOi42Zn0iCiAgICAgICAgYXNzZXJ0IGFicyhvdXJfYWNjIC0gZXZhbF9hY2MpIDwgdG9sLCBmIkFDQyBkcmlmdCB2cyBFdmFsdWF0b3I6IG91cnM9e291cl9hY2M6LjZmfSByZWY9e2V2YWxfYWNjOi42Zn0iCgogICAgaWYgdmVyYm9zZToKICAgICAgICBwcmludChmIlttZXRyaWNzXSBhZ3JlZW1lbnQgT0sgIG91cnM9KHtvdXJfYXVjOi40Zn0se291cl9hY2M6LjRmfSkgIgogICAgICAgICAgICAgIGYiZ2V0WD0oe3JlZl9hdWM6LjRmfSx7cmVmX2FjYzouNGZ9KSIKICAgICAgICAgICAgICArIChmIiBFdmFsdWF0b3I9KHtldmFsX2F1YzouNGZ9LHtldmFsX2FjYzouNGZ9KSIgaWYgZXZhbF9hdWMgaXMgbm90IE5vbmUgZWxzZSAiIikpCiAgICByZXR1cm4gZGljdChhdWM9b3VyX2F1YywgYWNjPW91cl9hY2MpCg==',
    'metrics_test.py': 'IiIiU3RhbmRhbG9uZSBhZ3JlZW1lbnQgdGVzdDogb3VyIG1ldHJpY3MgdnMgdGhlIG1lZG1uaXN0IG9yYWNsZS4KClJ1biB3aXRoOiAgcHl0aG9uIC1tIHNyYy5tZXRyaWNzX3Rlc3QKVXNlcyBzeW50aGV0aWMgcHJlZGljdGlvbnMgb25seSAobm8gdG9yY2ggLyBubyBkYXRhc2V0IGRvd25sb2FkIG5lZWRlZCkuCkV4aXRzIG5vbi16ZXJvIGlmIGFueSBtZXRyaWMgZHJpZnRzIGJleW9uZCAxZS02IGZyb20gbWVkbW5pc3QncyBnZXRBVUMvZ2V0QUNDLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuIGltcG9ydCBtZXRyaWNzCgoKZGVmIG1haW4oKToKICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZSgwKQoKICAgICMgbXVsdGktY2xhc3MgKERlcm1hTU5JU1Q9NywgUGF0aE1OSVNUPTkpCiAgICBmb3IgQyBpbiAoNywgOSk6CiAgICAgICAgTiA9IDMwMDAKICAgICAgICB5X3RydWUgPSBybmcucmFuZGludCgwLCBDLCBzaXplPShOLCAxKSkKICAgICAgICBsb2dpdHMgPSBybmcucmFuZG4oTiwgQykgKyBucC5leWUoQylbeV90cnVlLnNxdWVlemUoKV0gKiAxLjUKICAgICAgICBlID0gbnAuZXhwKGxvZ2l0cyAtIGxvZ2l0cy5tYXgoMSwga2VlcGRpbXM9VHJ1ZSkpCiAgICAgICAgeV9zY29yZSA9IGUgLyBlLnN1bSgxLCBrZWVwZGltcz1UcnVlKQogICAgICAgIG1ldHJpY3MuY2hlY2tfYWdyZWVtZW50KHlfdHJ1ZSwgeV9zY29yZSwgIm11bHRpLWNsYXNzIiwgdG9sPTFlLTYpCgogICAgIyBtdWx0aS1sYWJlbCwgYmluYXJ5LWNsYXNzIChDaGVzdE1OSVNUPTE0KQogICAgTCwgTiA9IDE0LCAyMDAwCiAgICB5X3RydWUgPSAocm5nLnJhbmQoTiwgTCkgPiAwLjcpLmFzdHlwZShpbnQpCiAgICB5X3Njb3JlID0gcm5nLnJhbmQoTiwgTCkKICAgIG1ldHJpY3MuY2hlY2tfYWdyZWVtZW50KHlfdHJ1ZSwgeV9zY29yZSwgIm11bHRpLWxhYmVsLCBiaW5hcnktY2xhc3MiLCB0b2w9MWUtNikKCiAgICAjIGJpbmFyeS1jbGFzcwogICAgeV90cnVlID0gcm5nLnJhbmRpbnQoMCwgMiwgc2l6ZT0oTiwgMSkpCiAgICB5X3Njb3JlID0gcm5nLnJhbmQoTiwgMikKICAgIHlfc2NvcmUgPSB5X3Njb3JlIC8geV9zY29yZS5zdW0oMSwga2VlcGRpbXM9VHJ1ZSkKICAgIG1ldHJpY3MuY2hlY2tfYWdyZWVtZW50KHlfdHJ1ZSwgeV9zY29yZSwgImJpbmFyeS1jbGFzcyIsIHRvbD0xZS02KQoKICAgIHByaW50KCJBTEwgTUVUUklDIEFHUkVFTUVOVCBURVNUUyBQQVNTRUQgKHRvbD0xZS02KSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=',
    'models.py': 'IiIiTW9kZWwgZGVmaW5pdGlvbnMsIHJlaW1wbGVtZW50ZWQgZnJvbSBzY3JhdGNoLgoKVHdvIGJhY2tib25lcywgY2hvc2VuIGJ5IGlucHV0IHJlc29sdXRpb24gKHRoaXMgaXMgZGVsaWJlcmF0ZSBhbmQgbWF0Y2hlcyB0aGUKTWVkTU5JU1QgdjIgcHJvdG9jb2wpOgoKKiBgYHNpemUgPT0gMjhgYCAgLT4gYSBDSUZBUi1zdHlsZSBSZXNOZXQ6IDN4MyBzdHJpZGUtMSBzdGVtLCAqbm8qIG1heC1wb29sLAogIGZvdXIgcmVzaWR1YWwgc3RhZ2VzIHdpdGggc3RyaWRlcyBgYFsxLCAyLCAyLCAyXWBgIGFuZCB3aWR0aHMKICBgYFs2NCwgMTI4LCAyNTYsIDUxMl1gYC4gUmVzTmV0LTE4IHVzZXMgYGBCYXNpY0Jsb2NrYGAgd2l0aCBgYFsyLDIsMiwyXWBgOwogIFJlc05ldC01MCB1c2VzIGBgQm90dGxlbmVja2BgIChleHBhbnNpb24gNCkgd2l0aCBgYFszLDQsNiwzXWBgLgoqIGBgc2l6ZSA9PSAyMjRgYCAtPiBgYHRvcmNodmlzaW9uLm1vZGVscy5yZXNuZXQxOC81MGBgIHdpdGggdGhlIHN0YW5kYXJkCiAgSW1hZ2VOZXQgN3g3IHN0cmlkZS0yIHN0ZW0gKyBtYXgtcG9vbC4KCkEgYGB3aWR0aF9tdWx0YGAga25vYiBnaXZlcyB0aGUgbGlnaHR3ZWlnaHQgMC41eCB2YXJpYW50ICh3aWR0aHMKYGBbMzIsNjQsMTI4LDI1Nl1gYCkgdXNlZCBieSB0aGUgZWZmaWNpZW5jeSBleHRlbnNpb24uCgpOb25lIG9mIHRoaXMgaXMgY29waWVkIGZyb20gYGBNZWRNTklTVC9leHBlcmltZW50c2BgIG9yIGBga3VhbmdsaXUvcHl0b3JjaC1jaWZhcmBgOwppdCBpcyB3cml0dGVuIHRvIHRoZSBzcGVjIGFib3ZlLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCmRlZiBjb252M3gzKGluX3BsYW5lcywgb3V0X3BsYW5lcywgc3RyaWRlPTEpOgogICAgcmV0dXJuIG5uLkNvbnYyZChpbl9wbGFuZXMsIG91dF9wbGFuZXMsIGtlcm5lbF9zaXplPTMsIHN0cmlkZT1zdHJpZGUsCiAgICAgICAgICAgICAgICAgICAgIHBhZGRpbmc9MSwgYmlhcz1GYWxzZSkKCgpkZWYgY29udjF4MShpbl9wbGFuZXMsIG91dF9wbGFuZXMsIHN0cmlkZT0xKToKICAgIHJldHVybiBubi5Db252MmQoaW5fcGxhbmVzLCBvdXRfcGxhbmVzLCBrZXJuZWxfc2l6ZT0xLCBzdHJpZGU9c3RyaWRlLCBiaWFzPUZhbHNlKQoKCmNsYXNzIEJhc2ljQmxvY2sobm4uTW9kdWxlKToKICAgIGV4cGFuc2lvbiA9IDEKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fcGxhbmVzLCBwbGFuZXMsIHN0cmlkZT0xLCBkb3duc2FtcGxlPU5vbmUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuY29udjEgPSBjb252M3gzKGluX3BsYW5lcywgcGxhbmVzLCBzdHJpZGUpCiAgICAgICAgc2VsZi5ibjEgPSBubi5CYXRjaE5vcm0yZChwbGFuZXMpCiAgICAgICAgc2VsZi5jb252MiA9IGNvbnYzeDMocGxhbmVzLCBwbGFuZXMpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0yZChwbGFuZXMpCiAgICAgICAgc2VsZi5kb3duc2FtcGxlID0gZG93bnNhbXBsZQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIGlkZW50aXR5ID0geAogICAgICAgIG91dCA9IEYucmVsdShzZWxmLmJuMShzZWxmLmNvbnYxKHgpKSwgaW5wbGFjZT1UcnVlKQogICAgICAgIG91dCA9IHNlbGYuYm4yKHNlbGYuY29udjIob3V0KSkKICAgICAgICBpZiBzZWxmLmRvd25zYW1wbGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlkZW50aXR5ID0gc2VsZi5kb3duc2FtcGxlKHgpCiAgICAgICAgcmV0dXJuIEYucmVsdShvdXQgKyBpZGVudGl0eSwgaW5wbGFjZT1UcnVlKQoKCmNsYXNzIEJvdHRsZW5lY2sobm4uTW9kdWxlKToKICAgIGV4cGFuc2lvbiA9IDQKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fcGxhbmVzLCBwbGFuZXMsIHN0cmlkZT0xLCBkb3duc2FtcGxlPU5vbmUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuY29udjEgPSBjb252MXgxKGluX3BsYW5lcywgcGxhbmVzKQogICAgICAgIHNlbGYuYm4xID0gbm4uQmF0Y2hOb3JtMmQocGxhbmVzKQogICAgICAgIHNlbGYuY29udjIgPSBjb252M3gzKHBsYW5lcywgcGxhbmVzLCBzdHJpZGUpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0yZChwbGFuZXMpCiAgICAgICAgc2VsZi5jb252MyA9IGNvbnYxeDEocGxhbmVzLCBwbGFuZXMgKiBzZWxmLmV4cGFuc2lvbikKICAgICAgICBzZWxmLmJuMyA9IG5uLkJhdGNoTm9ybTJkKHBsYW5lcyAqIHNlbGYuZXhwYW5zaW9uKQogICAgICAgIHNlbGYuZG93bnNhbXBsZSA9IGRvd25zYW1wbGUKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBpZGVudGl0eSA9IHgKICAgICAgICBvdXQgPSBGLnJlbHUoc2VsZi5ibjEoc2VsZi5jb252MSh4KSksIGlucGxhY2U9VHJ1ZSkKICAgICAgICBvdXQgPSBGLnJlbHUoc2VsZi5ibjIoc2VsZi5jb252MihvdXQpKSwgaW5wbGFjZT1UcnVlKQogICAgICAgIG91dCA9IHNlbGYuYm4zKHNlbGYuY29udjMob3V0KSkKICAgICAgICBpZiBzZWxmLmRvd25zYW1wbGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlkZW50aXR5ID0gc2VsZi5kb3duc2FtcGxlKHgpCiAgICAgICAgcmV0dXJuIEYucmVsdShvdXQgKyBpZGVudGl0eSwgaW5wbGFjZT1UcnVlKQoKCmNsYXNzIENpZmFyUmVzTmV0KG5uLk1vZHVsZSk6CiAgICAiIiJDSUZBUi1zdHlsZSBSZXNOZXQgZm9yIDI4eDI4IGlucHV0cyAoM3gzIHN0ZW0sIG5vIG1heC1wb29sKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgYmxvY2ssIGxheWVycywgbnVtX2NsYXNzZXMsIGluX2NoYW5uZWxzPTMsIHdpZHRoX211bHQ9MS4wKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICB3aWR0aHMgPSBbaW50KHcgKiB3aWR0aF9tdWx0KSBmb3IgdyBpbiAoNjQsIDEyOCwgMjU2LCA1MTIpXQogICAgICAgIHNlbGYuaW5fcGxhbmVzID0gd2lkdGhzWzBdCgogICAgICAgIHNlbGYuY29udjEgPSBubi5Db252MmQoaW5fY2hhbm5lbHMsIHdpZHRoc1swXSwga2VybmVsX3NpemU9Mywgc3RyaWRlPTEsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwYWRkaW5nPTEsIGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5ibjEgPSBubi5CYXRjaE5vcm0yZCh3aWR0aHNbMF0pCgogICAgICAgIHNlbGYubGF5ZXIxID0gc2VsZi5fbWFrZV9sYXllcihibG9jaywgd2lkdGhzWzBdLCBsYXllcnNbMF0sIHN0cmlkZT0xKQogICAgICAgIHNlbGYubGF5ZXIyID0gc2VsZi5fbWFrZV9sYXllcihibG9jaywgd2lkdGhzWzFdLCBsYXllcnNbMV0sIHN0cmlkZT0yKQogICAgICAgIHNlbGYubGF5ZXIzID0gc2VsZi5fbWFrZV9sYXllcihibG9jaywgd2lkdGhzWzJdLCBsYXllcnNbMl0sIHN0cmlkZT0yKQogICAgICAgIHNlbGYubGF5ZXI0ID0gc2VsZi5fbWFrZV9sYXllcihibG9jaywgd2lkdGhzWzNdLCBsYXllcnNbM10sIHN0cmlkZT0yKQoKICAgICAgICBzZWxmLmF2Z3Bvb2wgPSBubi5BZGFwdGl2ZUF2Z1Bvb2wyZCgoMSwgMSkpCiAgICAgICAgc2VsZi5mYyA9IG5uLkxpbmVhcih3aWR0aHNbM10gKiBibG9jay5leHBhbnNpb24sIG51bV9jbGFzc2VzKQoKICAgICAgICBmb3IgbSBpbiBzZWxmLm1vZHVsZXMoKToKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShtLCBubi5Db252MmQpOgogICAgICAgICAgICAgICAgbm4uaW5pdC5rYWltaW5nX25vcm1hbF8obS53ZWlnaHQsIG1vZGU9ImZhbl9vdXQiLCBub25saW5lYXJpdHk9InJlbHUiKQogICAgICAgICAgICBlbGlmIGlzaW5zdGFuY2UobSwgbm4uQmF0Y2hOb3JtMmQpOgogICAgICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8obS53ZWlnaHQsIDEpCiAgICAgICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XyhtLmJpYXMsIDApCgogICAgZGVmIF9tYWtlX2xheWVyKHNlbGYsIGJsb2NrLCBwbGFuZXMsIGJsb2Nrcywgc3RyaWRlKToKICAgICAgICBkb3duc2FtcGxlID0gTm9uZQogICAgICAgIGlmIHN0cmlkZSAhPSAxIG9yIHNlbGYuaW5fcGxhbmVzICE9IHBsYW5lcyAqIGJsb2NrLmV4cGFuc2lvbjoKICAgICAgICAgICAgZG93bnNhbXBsZSA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgICAgICBjb252MXgxKHNlbGYuaW5fcGxhbmVzLCBwbGFuZXMgKiBibG9jay5leHBhbnNpb24sIHN0cmlkZSksCiAgICAgICAgICAgICAgICBubi5CYXRjaE5vcm0yZChwbGFuZXMgKiBibG9jay5leHBhbnNpb24pLAogICAgICAgICAgICApCiAgICAgICAgbGF5ZXJzID0gW2Jsb2NrKHNlbGYuaW5fcGxhbmVzLCBwbGFuZXMsIHN0cmlkZSwgZG93bnNhbXBsZSldCiAgICAgICAgc2VsZi5pbl9wbGFuZXMgPSBwbGFuZXMgKiBibG9jay5leHBhbnNpb24KICAgICAgICBmb3IgXyBpbiByYW5nZSgxLCBibG9ja3MpOgogICAgICAgICAgICBsYXllcnMuYXBwZW5kKGJsb2NrKHNlbGYuaW5fcGxhbmVzLCBwbGFuZXMpKQogICAgICAgIHJldHVybiBubi5TZXF1ZW50aWFsKCpsYXllcnMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeCA9IEYucmVsdShzZWxmLmJuMShzZWxmLmNvbnYxKHgpKSwgaW5wbGFjZT1UcnVlKQogICAgICAgIHggPSBzZWxmLmxheWVyMSh4KQogICAgICAgIHggPSBzZWxmLmxheWVyMih4KQogICAgICAgIHggPSBzZWxmLmxheWVyMyh4KQogICAgICAgIHggPSBzZWxmLmxheWVyNCh4KQogICAgICAgIHggPSBzZWxmLmF2Z3Bvb2woeCkKICAgICAgICB4ID0gdG9yY2guZmxhdHRlbih4LCAxKQogICAgICAgIHJldHVybiBzZWxmLmZjKHgpCgoKZGVmIGJ1aWxkX21vZGVsKG1vZGVsX25hbWUsIHNpemUsIG51bV9jbGFzc2VzLCBpbl9jaGFubmVscz0zLCB3aWR0aF9tdWx0PTEuMCk6CiAgICAiIiJCdWlsZCBhIG1vZGVsIGZvciB0aGUgZ2l2ZW4gYXJjaGl0ZWN0dXJlIC8gcmVzb2x1dGlvbi4KCiAgICBBcmdzOgogICAgICAgIG1vZGVsX25hbWU6IGBgInJlc25ldDE4ImBgIG9yIGBgInJlc25ldDUwImBgLgogICAgICAgIHNpemU6IDI4IChDSUZBUi1zdHlsZSkgb3IgMjI0ICh0b3JjaHZpc2lvbiBJbWFnZU5ldCBiYWNrYm9uZSkuCiAgICAgICAgbnVtX2NsYXNzZXM6IG51bWJlciBvZiBvdXRwdXQgbG9naXRzLgogICAgICAgIGluX2NoYW5uZWxzOiBpbnB1dCBjaGFubmVscyAoMyBmb3IgdGhlIFJHQiBNZWRNTklTVCBkYXRhc2V0cyB3ZSB1c2UpLgogICAgICAgIHdpZHRoX211bHQ6IGNoYW5uZWwgbXVsdGlwbGllcjsgMC41IGdpdmVzIHRoZSBsaWdodHdlaWdodCB2YXJpYW50LgogICAgICAgICAgICAgICAgICAgIE9ubHkgc3VwcG9ydGVkIGF0IHNpemUgMjguCiAgICAiIiIKICAgIG1vZGVsX25hbWUgPSBtb2RlbF9uYW1lLmxvd2VyKCkKICAgIGlmIG1vZGVsX25hbWUgbm90IGluICgicmVzbmV0MTgiLCAicmVzbmV0NTAiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBtb2RlbCB7bW9kZWxfbmFtZSFyfSIpCgogICAgaWYgc2l6ZSA9PSAyODoKICAgICAgICBpZiBtb2RlbF9uYW1lID09ICJyZXNuZXQxOCI6CiAgICAgICAgICAgIHJldHVybiBDaWZhclJlc05ldChCYXNpY0Jsb2NrLCBbMiwgMiwgMiwgMl0sIG51bV9jbGFzc2VzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaW5fY2hhbm5lbHM9aW5fY2hhbm5lbHMsIHdpZHRoX211bHQ9d2lkdGhfbXVsdCkKICAgICAgICByZXR1cm4gQ2lmYXJSZXNOZXQoQm90dGxlbmVjaywgWzMsIDQsIDYsIDNdLCBudW1fY2xhc3NlcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgaW5fY2hhbm5lbHM9aW5fY2hhbm5lbHMsIHdpZHRoX211bHQ9d2lkdGhfbXVsdCkKCiAgICBpZiBzaXplID09IDIyNDoKICAgICAgICBpZiB3aWR0aF9tdWx0ICE9IDEuMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigid2lkdGhfbXVsdCBpcyBvbmx5IHN1cHBvcnRlZCBmb3IgdGhlIHNpemUtMjggQ0lGQVIgYmFja2JvbmUiKQogICAgICAgIGlmIGluX2NoYW5uZWxzICE9IDM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRvcmNodmlzaW9uIHJlc25ldCBiYWNrYm9uZSBleHBlY3RzIDMgaW5wdXQgY2hhbm5lbHMiKQogICAgICAgIGltcG9ydCB0b3JjaHZpc2lvbi5tb2RlbHMgYXMgdHZtCgogICAgICAgIHRyeTogICMgbmV3ZXIgdG9yY2h2aXNpb24KICAgICAgICAgICAgY3RvciA9IGdldGF0dHIodHZtLCBtb2RlbF9uYW1lKQogICAgICAgICAgICByZXR1cm4gY3Rvcih3ZWlnaHRzPU5vbmUsIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzKQogICAgICAgIGV4Y2VwdCBUeXBlRXJyb3I6ICAjIG9sZGVyIHRvcmNodmlzaW9uIHdpdGhvdXQgYHdlaWdodHM9YAogICAgICAgICAgICBjdG9yID0gZ2V0YXR0cih0dm0sIG1vZGVsX25hbWUpCiAgICAgICAgICAgIHJldHVybiBjdG9yKHByZXRyYWluZWQ9RmFsc2UsIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzKQoKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bnN1cHBvcnRlZCBzaXplIHtzaXplfTsgZXhwZWN0ZWQgMjggb3IgMjI0IikKCgpkZWYgY291bnRfcGFyYW1ldGVycyhtb2RlbCk6CiAgICByZXR1cm4gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKQoKCmRlZiBtb2RlbF9zaXplX21iKG1vZGVsKToKICAgICIiIk9uLWRpc2sgc2l6ZSBpbiBNQiBvZiB0aGUgbW9kZWwncyBmbG9hdDMyIHN0YXRlIGRpY3QuIiIiCiAgICBuX2J5dGVzID0gc3VtKHAubnVtZWwoKSAqIHAuZWxlbWVudF9zaXplKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKQogICAgbl9ieXRlcyArPSBzdW0oYi5udW1lbCgpICogYi5lbGVtZW50X3NpemUoKSBmb3IgYiBpbiBtb2RlbC5idWZmZXJzKCkpCiAgICByZXR1cm4gbl9ieXRlcyAvICgxMDI0ICoqIDIpCg==',
    'plotting.py': 'IiIiU2hhcmVkIHBsb3R0aW5nIHN0eWxlIGZvciBldmVyeSBmaWd1cmUgaW4gdGhlIHBhcGVyLgoKT25lIHBsYWNlIGZvciB0aGUgY29sb3JibGluZC1zYWZlIHBhbGV0dGUsIGNsZWFuIG1hdHBsb3RsaWIgZGVmYXVsdHMgKG5vIHRvcC8KcmlnaHQgc3BpbmVzLCByZWFkYWJsZSBmb250cyksIGFuZCBzaW5nbGUtY29sdW1uIHNpemluZyAofjMuMyBpbiB3aWRlKS4gRXZlcnkKZmlndXJlIHNjcmlwdCBpbXBvcnRzIGZyb20gaGVyZSBhbmQgc2F2ZXMgdGhyb3VnaCA6ZnVuYzpgc2F2ZWZpZ2AsIHdoaWNoIHdyaXRlcwpib3RoIGEgdmVjdG9yIFBERiBhbmQgYSAzMDAtZHBpIFBORyBpbnRvIGBgcmVwb3J0L2ZpZ3VyZXMvYGAuIFJlZ2VuZXJhdGluZwpmaWd1cmVzIG5ldmVyIHJlLXJ1bnMgdHJhaW5pbmcg4oCUIHRoZSBmaWd1cmUgc2NyaXB0cyByZWFkIG9ubHkgZnJvbSBgYHJlc3VsdHMvYGAuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCgppbXBvcnQgbWF0cGxvdGxpYgptYXRwbG90bGliLnVzZSgiQWdnIikKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAoKCiMgT2thYmUtSXRvIGNvbG9yYmxpbmQtc2FmZSBxdWFsaXRhdGl2ZSBwYWxldHRlICg4IGh1ZXMsIGRldXRlcmFub3BpYS1zYWZlKS4KUEFMRVRURSA9IFsKICAgICIjMDA3MkIyIiwgICMgYmx1ZQogICAgIiNFNjlGMDAiLCAgIyBvcmFuZ2UKICAgICIjMDA5RTczIiwgICMgZ3JlZW4KICAgICIjRDU1RTAwIiwgICMgdmVybWlsbGlvbgogICAgIiNDQzc5QTciLCAgIyByZWRkaXNoIHB1cnBsZQogICAgIiM1NkI0RTkiLCAgIyBza3kgYmx1ZQogICAgIiNGMEU0NDIiLCAgIyB5ZWxsb3cKICAgICIjMDAwMDAwIiwgICMgYmxhY2sKXQoKIyBEaXZlcmdpbmcgY29sb3JtYXAgZm9yIHNpZ25lZCAob3VycyAtIHBhcGVyKSBkZWx0YXMsIGNlbnRlcmVkIGF0IHplcm8uCkRJVkVSR0lOR19DTUFQID0gIlJkQnVfciIKCiMgU2luZ2xlLWNvbHVtbiBmaWd1cmUgd2lkdGggKGluY2hlcykgZm9yIGEgdHdvLWNvbHVtbiBwYXBlci4KQ09MX1dJRFRIID0gMy4zCkdPTERFTiA9IDAuNjIgICMgaGVpZ2h0OndpZHRoIHJhdGlvIHVzZWQgZm9yIGRlZmF1bHQgc2luZ2xlLXBhbmVsIGZpZ3VyZXMKCgpkZWYgc2V0X3N0eWxlKCk6CiAgICAiIiJJbnN0YWxsIHRoZSBzaGFyZWQgcmNQYXJhbXMuIElkZW1wb3RlbnQ7IGNhbGwgb25jZSBiZWZvcmUgcGxvdHRpbmcuIiIiCiAgICBwbHQucmNQYXJhbXMudXBkYXRlKHsKICAgICAgICAiZmlndXJlLmRwaSI6IDExMCwKICAgICAgICAic2F2ZWZpZy5kcGkiOiAzMDAsCiAgICAgICAgInNhdmVmaWcuYmJveCI6ICJ0aWdodCIsCiAgICAgICAgImZvbnQuc2l6ZSI6IDgsCiAgICAgICAgImF4ZXMudGl0bGVzaXplIjogOSwKICAgICAgICAiYXhlcy5sYWJlbHNpemUiOiA4LAogICAgICAgICJsZWdlbmQuZm9udHNpemUiOiA3LAogICAgICAgICJ4dGljay5sYWJlbHNpemUiOiA3LAogICAgICAgICJ5dGljay5sYWJlbHNpemUiOiA3LAogICAgICAgICJheGVzLmxpbmV3aWR0aCI6IDAuOCwKICAgICAgICAiYXhlcy5zcGluZXMudG9wIjogRmFsc2UsCiAgICAgICAgImF4ZXMuc3BpbmVzLnJpZ2h0IjogRmFsc2UsCiAgICAgICAgImF4ZXMuZ3JpZCI6IFRydWUsCiAgICAgICAgImdyaWQuYWxwaGEiOiAwLjI1LAogICAgICAgICJncmlkLmxpbmV3aWR0aCI6IDAuNiwKICAgICAgICAibGluZXMubGluZXdpZHRoIjogMS41LAogICAgICAgICJsZWdlbmQuZnJhbWVvbiI6IEZhbHNlLAogICAgICAgICJmaWd1cmUuYXV0b2xheW91dCI6IEZhbHNlLAogICAgfSkKICAgICMgQ3ljbGUgdGhlIGNvbG9yYmxpbmQtc2FmZSBwYWxldHRlIGJ5IGRlZmF1bHQuCiAgICBwbHQucmNQYXJhbXNbImF4ZXMucHJvcF9jeWNsZSJdID0gcGx0LmN5Y2xlcihjb2xvcj1QQUxFVFRFKQoKCmRlZiBuZXdfZmlnKHdpZHRoPUNPTF9XSURUSCwgaGVpZ2h0PU5vbmUsIG5jb2xzPTEsIG5yb3dzPTEsICoqa3cpOgogICAgIiIiUmV0dXJuIGBgKGZpZywgYXgpYGAgc2l6ZWQgZm9yIHRoZSBwYXBlci4gYGBoZWlnaHRgYCBkZWZhdWx0cyB0byBnb2xkZW4uIiIiCiAgICBpZiBoZWlnaHQgaXMgTm9uZToKICAgICAgICBoZWlnaHQgPSB3aWR0aCAqIEdPTERFTgogICAgcmV0dXJuIHBsdC5zdWJwbG90cyhucm93cywgbmNvbHMsIGZpZ3NpemU9KHdpZHRoICogbmNvbHMsIGhlaWdodCAqIG5yb3dzKSwgKiprdykKCgpkZWYgZGVzcGluZShheCk6CiAgICAiIiJSZW1vdmUgdGhlIHRvcC9yaWdodCBzcGluZXMgb24gYSBzaW5nbGUgQXhlcyAocmNQYXJhbXMgaGFuZGxlcyBtb3N0KS4iIiIKICAgIGF4LnNwaW5lc1sidG9wIl0uc2V0X3Zpc2libGUoRmFsc2UpCiAgICBheC5zcGluZXNbInJpZ2h0Il0uc2V0X3Zpc2libGUoRmFsc2UpCgoKZGVmIHNhdmVmaWcoZmlnLCBuYW1lLCByZXBvcnRfZGlyPSJyZXBvcnQiKToKICAgICIiIlNhdmUgYGBmaWdgYCBhcyBib3RoIFBERiBhbmQgMzAwLWRwaSBQTkcgdW5kZXIgYGByZXBvcnRfZGlyL2ZpZ3VyZXMvYGAuCgogICAgUmV0dXJucyB0aGUgUE5HIHBhdGggKGhhbmR5IGZvciBub3RlYm9vayBkaXNwbGF5KS4gYGBuYW1lYGAgaXMgYSBzdGVtIHdpdGgKICAgIG5vIGV4dGVuc2lvbiwgZS5nLiBgYCJmaWcxX3JlcHJvZHVjdGlvbiJgYC4KICAgICIiIgogICAgZmlnX2RpciA9IG9zLnBhdGguam9pbihyZXBvcnRfZGlyLCAiZmlndXJlcyIpCiAgICBvcy5tYWtlZGlycyhmaWdfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgcGRmID0gb3MucGF0aC5qb2luKGZpZ19kaXIsIGYie25hbWV9LnBkZiIpCiAgICBwbmcgPSBvcy5wYXRoLmpvaW4oZmlnX2RpciwgZiJ7bmFtZX0ucG5nIikKICAgIGZpZy5zYXZlZmlnKHBkZikKICAgIGZpZy5zYXZlZmlnKHBuZywgZHBpPTMwMCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICByZXR1cm4gcG5nCg==',
    'reference.py': 'IiIiUGFwZXIgcmVmZXJlbmNlIG51bWJlcnMgKE1lZE1OSVNUIHYyLCBZYW5nIGV0IGFsLiAyMDIzLCBUYWJsZSAzKS4KCktlcHQgZGVwZW5kZW5jeS1mcmVlIHNvIGFnZ3JlZ2F0aW9uIGNhbiBpbXBvcnQgaXQgd2l0aG91dCB0b3JjaC4KRm9ybWF0OiAoZGF0YXNldCwgbW9kZWwsIHNpemUpIC0+IChBVUMsIEFDQykuCiIiIgoKUkVGRVJFTkNFID0gewogICAgIyBEZXJtYU1OSVNUIOKAlCB0aGUgcHJpbWFyeSwgaW4tZGVwdGggZm91ci1jb25maWcgbWF0cml4IChSMTgvUjUwIHggMjgvMjI0KS4KICAgICgiZGVybWFtbmlzdCIsICJyZXNuZXQxOCIsIDI4KTogKDAuOTE3LCAwLjczNSksCiAgICAoImRlcm1hbW5pc3QiLCAicmVzbmV0MTgiLCAyMjQpOiAoMC45MjAsIDAuNzU0KSwKICAgICgiZGVybWFtbmlzdCIsICJyZXNuZXQ1MCIsIDI4KTogKDAuOTEzLCAwLjczNSksCiAgICAoImRlcm1hbW5pc3QiLCAicmVzbmV0NTAiLCAyMjQpOiAoMC45MTIsIDAuNzMxKSwKICAgICMgVGhlIG90aGVyIGVsZXZlbiBNZWRNTklTVDJEIGRhdGFzZXRzIOKAlCBSZXNOZXQtMTggQCAyOCBvbmx5IChvdXIgc3dlZXApLgogICAgKCJwYXRobW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjk4MywgMC45MDcpLAogICAgKCJjaGVzdG1uaXN0IiwgInJlc25ldDE4IiwgMjgpOiAoMC43NjgsIDAuOTQ3KSwKICAgICgib2N0bW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjk0MywgMC43NDMpLAogICAgKCJwbmV1bW9uaWFtbmlzdCIsICJyZXNuZXQxOCIsIDI4KTogKDAuOTQ0LCAwLjg1NCksCiAgICAoInJldGluYW1uaXN0IiwgInJlc25ldDE4IiwgMjgpOiAoMC43MTcsIDAuNTI0KSwKICAgICgiYnJlYXN0bW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjkwMSwgMC44NjMpLAogICAgKCJibG9vZG1uaXN0IiwgInJlc25ldDE4IiwgMjgpOiAoMC45OTgsIDAuOTU4KSwKICAgICgidGlzc3VlbW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjkzMCwgMC42NzYpLAogICAgKCJvcmdhbmFtbmlzdCIsICJyZXNuZXQxOCIsIDI4KTogKDAuOTk3LCAwLjkzNSksCiAgICAoIm9yZ2FuY21uaXN0IiwgInJlc25ldDE4IiwgMjgpOiAoMC45OTIsIDAuOTAwKSwKICAgICgib3JnYW5zbW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjk3MiwgMC43ODIpLAogICAgIyBFeHRyYSBSMTgvUjUwIEAgMjI0LzI4IFBhdGhNTklTVCByZWZlcmVuY2VzIChrZXB0IGZvciBvcHRpb25hbCBkZXB0aCBydW5zKS4KICAgICgicGF0aG1uaXN0IiwgInJlc25ldDE4IiwgMjI0KTogKDAuOTg5LCAwLjkwOSksCiAgICAoInBhdGhtbmlzdCIsICJyZXNuZXQ1MCIsIDI4KTogKDAuOTkwLCAwLjkxMSksCiAgICAoInBhdGhtbmlzdCIsICJyZXNuZXQ1MCIsIDIyNCk6ICgwLjk4OSwgMC44OTIpLAp9CgojIENvbnZlbmllbmNlOiB0aGUgdHdlbHZlIE1lZE1OSVNUMkQgZGF0YXNldHMgaW4gdGhlIHNwZWMncyBjYW5vbmljYWwgb3JkZXIsCiMgZWFjaCB3aXRoIHRoZSBSZXNOZXQtMTggQCAyOCByZWZlcmVuY2UgdGhlIHJlcGxpY2F0aW9uIHN3ZWVwIHRhcmdldHMuCkRBVEFTRVRTXzJEID0gWwogICAgInBhdGhtbmlzdCIsICJjaGVzdG1uaXN0IiwgImRlcm1hbW5pc3QiLCAib2N0bW5pc3QiLAogICAgInBuZXVtb25pYW1uaXN0IiwgInJldGluYW1uaXN0IiwgImJyZWFzdG1uaXN0IiwgImJsb29kbW5pc3QiLAogICAgInRpc3N1ZW1uaXN0IiwgIm9yZ2FuYW1uaXN0IiwgIm9yZ2FuY21uaXN0IiwgIm9yZ2Fuc21uaXN0IiwKXQo=',
    'run.py': 'IiIiQ29uZmlnLWRyaXZlbiBlbnRyeSBwb2ludCBmb3IgYSBzaW5nbGUgdHJhaW5pbmcgcnVuLgoKVXNhZ2UgKENMSSk6CiAgICBweXRob24gLW0gc3JjLnJ1biAtLWRhdGFzZXQgZGVybWFtbmlzdCAtLW1vZGVsIHJlc25ldDE4IC0tc2l6ZSAyOCAtLXNlZWQgMCBcCiAgICAgICAgLS1lcG9jaHMgMTAwIC0tcmVzdWx0cy1kaXIgcmVzdWx0cwoKT3IgaW1wb3J0IDpmdW5jOmBydW5fY29uZmlnYCBmcm9tIGEgbm90ZWJvb2suCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBpbwppbXBvcnQgc3lzCmltcG9ydCBqc29uCmltcG9ydCB0aW1lCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgcGxhdGZvcm0KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBhc2RpY3QsIGZpZWxkCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIC4gaW1wb3J0IGRhdGEgYXMgZGF0YW1vZApmcm9tIC4gaW1wb3J0IG1ldHJpY3MgYXMgbWV0cmljc21vZApmcm9tIC5tb2RlbHMgaW1wb3J0IGJ1aWxkX21vZGVsLCBjb3VudF9wYXJhbWV0ZXJzLCBtb2RlbF9zaXplX21iCmZyb20gLnRyYWluIGltcG9ydCBzZXRfc2VlZCwgcnVuX3RyYWluaW5nCmZyb20gLmV2YWx1YXRlIGltcG9ydCBwcmVkaWN0LCBzYXZlX3ByZWRpY3Rpb25zLCBwcmVkaWN0aW9uX2ZpbGVuYW1lCmZyb20gLnJlZmVyZW5jZSBpbXBvcnQgUkVGRVJFTkNFCgoKQGRhdGFjbGFzcwpjbGFzcyBSdW5Db25maWc6CiAgICBkYXRhc2V0OiBzdHIgPSAiZGVybWFtbmlzdCIKICAgIG1vZGVsOiBzdHIgPSAicmVzbmV0MTgiCiAgICBzaXplOiBpbnQgPSAyOAogICAgc2VlZDogaW50ID0gMAogICAgZXBvY2hzOiBpbnQgPSAxMDAKICAgIGJhdGNoX3NpemU6IGludCA9IDEyOAogICAgbHI6IGZsb2F0ID0gMWUtMwogICAgYW1wOiBib29sID0gRmFsc2UKICAgIGRldGVybWluaXN0aWM6IGJvb2wgPSBUcnVlCiAgICB3aWR0aF9tdWx0OiBmbG9hdCA9IDEuMAogICAgIyBiaWFzLW1pdGlnYXRpb24gZmxhZ3MgKGV4dGVuc2lvbnMpCiAgICB3ZWlnaHRlZF9zYW1wbGVyOiBib29sID0gRmFsc2UKICAgIHdlaWdodGVkX2xvc3M6IGJvb2wgPSBGYWxzZQogICAgbnVtX3dvcmtlcnM6IGludCA9IDIKICAgIGNrcHRfZXZlcnk6IGludCA9IDUKICAgIHJlc3VtZTogYm9vbCA9IFRydWUKICAgIGRvd25sb2FkOiBib29sID0gVHJ1ZQogICAgcm9vdDogc3RyID0gTm9uZQogICAgcmVzdWx0c19kaXI6IHN0ciA9ICJyZXN1bHRzIgogICAgdGFnOiBzdHIgPSAiIiAgIyBleHRyYSBsYWJlbCBmb2xkZWQgaW50byB0aGUgcnVuIGRpcmVjdG9yeSBuYW1lCgogICAgZGVmIGZsYWcoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYuZGF0YXNldAoKICAgIGRlZiBydW5fbmFtZShzZWxmKToKICAgICAgICBwYXJ0cyA9IFtzZWxmLmRhdGFzZXQsIHNlbGYubW9kZWwsIGYic3tzZWxmLnNpemV9IiwgZiJzZWVke3NlbGYuc2VlZH0iXQogICAgICAgIGlmIHNlbGYud2lkdGhfbXVsdCAhPSAxLjA6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmInd7c2VsZi53aWR0aF9tdWx0fSIpCiAgICAgICAgaWYgc2VsZi53ZWlnaHRlZF9zYW1wbGVyOgogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoIndzYW1wbGVyIikKICAgICAgICBpZiBzZWxmLndlaWdodGVkX2xvc3M6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZCgid2xvc3MiKQogICAgICAgIGlmIHNlbGYudGFnOgogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoc2VsZi50YWcpCiAgICAgICAgcmV0dXJuICJfIi5qb2luKHBhcnRzKQoKCmRlZiBfdmVyc2lvbnMoKToKICAgIGltcG9ydCBza2xlYXJuCiAgICBpbXBvcnQgdG9yY2h2aXNpb24KICAgIGltcG9ydCBtZWRtbmlzdAogICAgcmV0dXJuIGRpY3QoCiAgICAgICAgcHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksCiAgICAgICAgdG9yY2g9dG9yY2guX192ZXJzaW9uX18sCiAgICAgICAgdG9yY2h2aXNpb249dG9yY2h2aXNpb24uX192ZXJzaW9uX18sCiAgICAgICAgbWVkbW5pc3Q9bWVkbW5pc3QuX192ZXJzaW9uX18sCiAgICAgICAgbnVtcHk9bnAuX192ZXJzaW9uX18sCiAgICAgICAgc2tsZWFybj1za2xlYXJuLl9fdmVyc2lvbl9fLAogICAgICAgIGN1ZGE9dG9yY2gudmVyc2lvbi5jdWRhLAogICAgICAgIGdwdT10b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIsCiAgICApCgoKZGVmIHJ1bl9jb25maWcoY2ZnOiBSdW5Db25maWcsIGxvZ19mbj1wcmludCwgdmVyaWZ5X21ldHJpY3M9VHJ1ZSk6CiAgICAiIiJFeGVjdXRlIG9uZSBydW4gZW5kLXRvLWVuZCBhbmQgd3JpdGUgYXJ0aWZhY3RzLiBSZXR1cm5zIGEgcmVzdWx0IGRpY3QuIiIiCiAgICBkZXZpY2UgPSAiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiCiAgICBpbmZvID0gZGF0YW1vZC5nZXRfaW5mbyhjZmcuZGF0YXNldCkKICAgIHRhc2sgPSBpbmZvWyJ0YXNrIl0KICAgIG5fY2xhc3NlcyA9IGxlbihpbmZvWyJsYWJlbCJdKQoKICAgIHVzZWRfZGV0ID0gc2V0X3NlZWQoY2ZnLnNlZWQsIGRldGVybWluaXN0aWM9Y2ZnLmRldGVybWluaXN0aWMpCiAgICBydW5fZGlyID0gb3MucGF0aC5qb2luKGNmZy5yZXN1bHRzX2RpciwgY2ZnLnJ1bl9uYW1lKCkpCiAgICBvcy5tYWtlZGlycyhydW5fZGlyLCBleGlzdF9vaz1UcnVlKQogICAgbG9nX2ZuKGYiPT09IHtjZmcucnVuX25hbWUoKX0gfCB0YXNrPXt0YXNrfSBjbGFzc2VzPXtuX2NsYXNzZXN9IGRldmljZT17ZGV2aWNlfSA9PT0iKQoKICAgIHNhbXBsZXIgPSBkYXRhbW9kLm1ha2Vfd2VpZ2h0ZWRfc2FtcGxlcihjZmcuZGF0YXNldCwgY2ZnLnJvb3QsIGNmZy5kb3dubG9hZCkgXAogICAgICAgIGlmIGNmZy53ZWlnaHRlZF9zYW1wbGVyIGVsc2UgTm9uZQogICAgbG9hZGVycyA9IGRhdGFtb2QuZ2V0X2xvYWRlcnMoCiAgICAgICAgY2ZnLmRhdGFzZXQsIGNmZy5zaXplLCBiYXRjaF9zaXplPWNmZy5iYXRjaF9zaXplLCByb290PWNmZy5yb290LAogICAgICAgIGRvd25sb2FkPWNmZy5kb3dubG9hZCwgbnVtX3dvcmtlcnM9Y2ZnLm51bV93b3JrZXJzLCBzYW1wbGVyPXNhbXBsZXIsCiAgICAgICAgcGluX21lbW9yeT0oZGV2aWNlID09ICJjdWRhIikpCgogICAgY2xhc3Nfd2VpZ2h0ID0gZGF0YW1vZC5jbGFzc193ZWlnaHRzKGNmZy5kYXRhc2V0LCBjZmcucm9vdCwgY2ZnLmRvd25sb2FkKSBcCiAgICAgICAgaWYgY2ZnLndlaWdodGVkX2xvc3MgZWxzZSBOb25lCgogICAgbW9kZWwgPSBidWlsZF9tb2RlbChjZmcubW9kZWwsIGNmZy5zaXplLCBuX2NsYXNzZXMsIGluX2NoYW5uZWxzPTMsCiAgICAgICAgICAgICAgICAgICAgICAgIHdpZHRoX211bHQ9Y2ZnLndpZHRoX211bHQpCiAgICBuX3BhcmFtcyA9IGNvdW50X3BhcmFtZXRlcnMobW9kZWwpCiAgICBzaXplX21iID0gbW9kZWxfc2l6ZV9tYihtb2RlbCkKICAgIGxvZ19mbihmIm1vZGVsIHBhcmFtcz17bl9wYXJhbXM6LH0gc2l6ZT17c2l6ZV9tYjouMmZ9IE1CIikKCiAgICBpZiBkZXZpY2UgPT0gImN1ZGEiOgogICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgcmVzdWx0ID0gcnVuX3RyYWluaW5nKAogICAgICAgIG1vZGVsLCBsb2FkZXJzLCB0YXNrLCBlcG9jaHM9Y2ZnLmVwb2NocywgbHI9Y2ZnLmxyLCBkZXZpY2U9ZGV2aWNlLAogICAgICAgIGNsYXNzX3dlaWdodD1jbGFzc193ZWlnaHQsIHVzZV9hbXA9Y2ZnLmFtcCwgZGV0ZXJtaW5pc3RpYz1jZmcuZGV0ZXJtaW5pc3RpYywKICAgICAgICBzZWVkPWNmZy5zZWVkLCBja3B0X2Rpcj1ydW5fZGlyLCBja3B0X2V2ZXJ5PWNmZy5ja3B0X2V2ZXJ5LAogICAgICAgIHJlc3VtZT1jZmcucmVzdW1lLCBsb2dfZm49bG9nX2ZuKQogICAgd2FsbCA9IHRpbWUudGltZSgpIC0gdDAKICAgIHBlYWtfZ3B1X21lbV9tYiA9ICh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkgLyAoMTAyNCAqKiAyKQogICAgICAgICAgICAgICAgICAgICAgIGlmIGRldmljZSA9PSAiY3VkYSIgZWxzZSBOb25lKQoKICAgICMgU2F2ZSB0ZXN0IHByZWRpY3Rpb25zIGluIHRoZSBtZWRtbmlzdC1jb252ZW50aW9uIGZpbGVuYW1lLgogICAgcHJlZF9uYW1lID0gcHJlZGljdGlvbl9maWxlbmFtZShjZmcuZmxhZygpLCAidGVzdCIsIHJlc3VsdFsidGVzdF9hdWMiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJ0ZXN0X2FjYyJdLCBjZmcuc2VlZCwgc2l6ZT1jZmcuc2l6ZSkKICAgIHNhdmVfcHJlZGljdGlvbnMocmVzdWx0WyJ0ZXN0X3lfc2NvcmUiXSwgb3MucGF0aC5qb2luKHJ1bl9kaXIsIHByZWRfbmFtZSkpCgogICAgIyBPcHRpb25hbCBvcmFjbGUgYWdyZWVtZW50IGNoZWNrIG9uIHRoZSByZWFsIHRlc3QgcHJlZGljdGlvbnMuCiAgICBhZ3JlZW1lbnQgPSBOb25lCiAgICBpZiB2ZXJpZnlfbWV0cmljczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGFncmVlbWVudCA9IG1ldHJpY3Ntb2QuY2hlY2tfYWdyZWVtZW50KAogICAgICAgICAgICAgICAgcmVzdWx0WyJ0ZXN0X3lfdHJ1ZSJdLCByZXN1bHRbInRlc3RfeV9zY29yZSJdLCB0YXNrLAogICAgICAgICAgICAgICAgZmxhZz1jZmcuZmxhZygpLCBzcGxpdD0idGVzdCIsIHNpemU9MjgsIHJvb3Q9Y2ZnLnJvb3QsCiAgICAgICAgICAgICAgICB0b2w9MWUtMywgdmVyYm9zZT1GYWxzZSkKICAgICAgICAgICAgbG9nX2ZuKGYiW3ZlcmlmeV0gbWV0cmljcyBhZ3JlZSB3aXRoIG1lZG1uaXN0LkV2YWx1YXRvciAoYXVjPXthZ3JlZW1lbnRbJ2F1YyddOi40Zn0pIikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAjIG5ldmVyIGxldCB2ZXJpZmljYXRpb24ga2lsbCBhIGNvbXBsZXRlZCBydW4KICAgICAgICAgICAgbG9nX2ZuKGYiW3ZlcmlmeV0gV0FSTklORyBhZ3JlZW1lbnQgY2hlY2sgc2tpcHBlZDoge2V9IikKCiAgICByZWYgPSBSRUZFUkVOQ0UuZ2V0KChjZmcuZGF0YXNldCwgY2ZnLm1vZGVsLCBjZmcuc2l6ZSkpCiAgICBydW5fanNvbiA9IGRpY3QoCiAgICAgICAgY29uZmlnPWFzZGljdChjZmcpLAogICAgICAgIHRhc2s9dGFzaywgbl9jbGFzc2VzPW5fY2xhc3NlcywKICAgICAgICBuX3BhcmFtcz1uX3BhcmFtcywgbW9kZWxfc2l6ZV9tYj1zaXplX21iLAogICAgICAgIGJlc3RfZXBvY2g9cmVzdWx0WyJiZXN0X2Vwb2NoIl0sIGJlc3RfdmFsX2F1Yz1yZXN1bHRbImJlc3RfdmFsX2F1YyJdLAogICAgICAgIHRyYWluX2F1Yz1yZXN1bHRbInRyYWluX2F1YyJdLCB0cmFpbl9hY2M9cmVzdWx0WyJ0cmFpbl9hY2MiXSwKICAgICAgICB2YWxfYXVjPXJlc3VsdFsidmFsX2F1YyJdLCB2YWxfYWNjPXJlc3VsdFsidmFsX2FjYyJdLAogICAgICAgIHRlc3RfYXVjPXJlc3VsdFsidGVzdF9hdWMiXSwgdGVzdF9hY2M9cmVzdWx0WyJ0ZXN0X2FjYyJdLAogICAgICAgIHJlZmVyZW5jZV9hdWM9KHJlZlswXSBpZiByZWYgZWxzZSBOb25lKSwKICAgICAgICByZWZlcmVuY2VfYWNjPShyZWZbMV0gaWYgcmVmIGVsc2UgTm9uZSksCiAgICAgICAgZGVsdGFfYXVjPShyZXN1bHRbInRlc3RfYXVjIl0gLSByZWZbMF0gaWYgcmVmIGVsc2UgTm9uZSksCiAgICAgICAgZGVsdGFfYWNjPShyZXN1bHRbInRlc3RfYWNjIl0gLSByZWZbMV0gaWYgcmVmIGVsc2UgTm9uZSksCiAgICAgICAgd2FsbF9jbG9ja19zPXdhbGwsCiAgICAgICAgcGVha19ncHVfbWVtX21iPXBlYWtfZ3B1X21lbV9tYiwKICAgICAgICBkZXRlcm1pbmlzdGljX2N1ZG5uPXVzZWRfZGV0LAogICAgICAgIGFtcD1jZmcuYW1wLAogICAgICAgIHNlZWQ9Y2ZnLnNlZWQsCiAgICAgICAgcHJlZGljdGlvbl9maWxlPXByZWRfbmFtZSwKICAgICAgICBoaXN0b3J5PXJlc3VsdFsiaGlzdG9yeSJdLAogICAgICAgIGFncmVlbWVudD1hZ3JlZW1lbnQsCiAgICAgICAgdmVyc2lvbnM9X3ZlcnNpb25zKCksCiAgICApCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKHJ1bl9kaXIsICJydW4uanNvbiIpLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHJ1bl9qc29uLCBmLCBpbmRlbnQ9MikKCiAgICBsb2dfZm4oZiJET05FIHtjZmcucnVuX25hbWUoKX0gdGVzdF9hdWM9e3Jlc3VsdFsndGVzdF9hdWMnXTouNGZ9ICIKICAgICAgICAgICBmInRlc3RfYWNjPXtyZXN1bHRbJ3Rlc3RfYWNjJ106LjRmfSAiCiAgICAgICAgICAgKyAoZiIocGFwZXIge3JlZlswXTouM2Z9L3tyZWZbMV06LjNmfSkiIGlmIHJlZiBlbHNlICIiKQogICAgICAgICAgICsgZiIgW3t3YWxsLzYwOi4xZn0gbWluXSIpCiAgICByZXR1cm4gcnVuX2pzb24KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgUnVuLW1hdHJpeCBlbnVtZXJhdGlvbiAodGllcnMpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKCiMgVGhlIGVsZXZlbiBub24tRGVybWEgTWVkTU5JU1QyRCBkYXRhc2V0cyBhcmUgc3BsaXQgYnkgZGF0YSBzY2FsZS4gVGhlIHNtYWxsIC8KIyBtZWRpdW0gb25lcyBnZXQgdGhyZWUgc2VlZHM7IHRoZSBmb3VyIGxhcmdlIG9uZXMgZ2V0IGEgc2luZ2xlIHNlZWQgKHNlZWQgMCksCiMgYmVjYXVzZSBhIHNpbmdsZSBydW4gaXMgc3RhYmxlIGF0IHRoYXQgZGF0YSBzY2FsZSBhbmQgdGhyZWUgd291bGQgdHJpcGxlIHRoZQojIGNvc3QuIFRoaXMgc3BsaXQgaXMgYSBjb21wdXRlIGRlY2lzaW9uLCBkaXNjbG9zZWQgaW4gdGhlIHdyaXRldXAuClNNQUxMX01FRElVTV9EQVRBU0VUUyA9ICgKICAgICJyZXRpbmFtbmlzdCIsICJicmVhc3RtbmlzdCIsICJwbmV1bW9uaWFtbmlzdCIsICJibG9vZG1uaXN0IiwKICAgICJvcmdhbmFtbmlzdCIsICJvcmdhbmNtbmlzdCIsICJvcmdhbnNtbmlzdCIsCikKTEFSR0VfREFUQVNFVFMgPSAoInRpc3N1ZW1uaXN0IiwgIm9jdG1uaXN0IiwgInBhdGhtbmlzdCIsICJjaGVzdG1uaXN0IikKCgpkZWYgcnVuX21hdHJpeChzZWVkcz0oMCwgMSwgMikpOgogICAgIiIiUmV0dXJuIHRoZSB0aWVyZWQgbGlzdCBvZiBgYCh0aWVyLCBkYXRhc2V0LCBtb2RlbCwgc2l6ZSwgc2VlZClgYCB0dXBsZXMuCgogICAgVGhpcyBpcyB0aGUgY29tcHV0ZS1taW5pbWFsIHNsaWNlIG9mIE1lZE1OSVNUIHYyIFRhYmxlIDMgdGhlIHJlcGxpY2F0aW9uCiAgICB0YXJnZXRzOgoKICAgICogKipUaWVyIDEqKiDigJQgRGVybWFNTklTVCAocHJpbWFyeSksIFJlc05ldC0xOC81MCB4IHNpemVzIDI4LzIyNCwgYWxsCiAgICAgIGBgc2VlZHNgYC4gVHdlbHZlIHJ1bnMgYXQgdGhyZWUgc2VlZHMuCiAgICAqICoqVGllciAyKiog4oCUIHRoZSBzbWFsbC9tZWRpdW0gZGF0YXNldHMgYXQgUmVzTmV0LTE4IEAgMjgsIGFsbCBgYHNlZWRzYGAuCiAgICAqICoqVGllciAzKiog4oCUIHRoZSBmb3VyIGxhcmdlIGRhdGFzZXRzIGF0IFJlc05ldC0xOCBAIDI4LCAqKnNlZWQgMCBvbmx5KioKICAgICAgKHNpbmdsZS1zZWVkIGZvciBjb21wdXRlIHJlYXNvbnM7IHN0YWJsZSBhdCB0aGF0IGRhdGEgc2NhbGUpLgogICAgIiIiCiAgICBtYXRyaXggPSBbXQogICAgIyBUaWVyIDE6IERlcm1hTU5JU1QsIGFsbCA0IG1vZGVsIHggc2l6ZSwgYWxsIHNlZWRzLgogICAgZm9yIG1vZGVsIGluICgicmVzbmV0MTgiLCAicmVzbmV0NTAiKToKICAgICAgICBmb3Igc2l6ZSBpbiAoMjgsIDIyNCk6CiAgICAgICAgICAgIGZvciBzIGluIHNlZWRzOgogICAgICAgICAgICAgICAgbWF0cml4LmFwcGVuZCgoMSwgImRlcm1hbW5pc3QiLCBtb2RlbCwgc2l6ZSwgcykpCiAgICAjIFRpZXIgMjogc21hbGwvbWVkaXVtIGRhdGFzZXRzLCBSZXNOZXQtMTggQCAyOCwgYWxsIHNlZWRzLgogICAgZm9yIGRhdGFzZXQgaW4gU01BTExfTUVESVVNX0RBVEFTRVRTOgogICAgICAgIGZvciBzIGluIHNlZWRzOgogICAgICAgICAgICBtYXRyaXguYXBwZW5kKCgyLCBkYXRhc2V0LCAicmVzbmV0MTgiLCAyOCwgcykpCiAgICAjIFRpZXIgMzogbGFyZ2UgZGF0YXNldHMsIFJlc05ldC0xOCBAIDI4LCBzaW5nbGUgc2VlZCAoc2VlZCAwKS4KICAgIGZvciBkYXRhc2V0IGluIExBUkdFX0RBVEFTRVRTOgogICAgICAgIG1hdHJpeC5hcHBlbmQoKDMsIGRhdGFzZXQsICJyZXNuZXQxOCIsIDI4LCAwKSkKICAgIHJldHVybiBtYXRyaXgKCgpkZWYgX3BhcnNlX2FyZ3MoYXJndj1Ob25lKToKICAgIHAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iTWVkTU5JU1QgdjIgcmVwbGljYXRpb246IHNpbmdsZSBydW4iKQogICAgcC5hZGRfYXJndW1lbnQoIi0tZGF0YXNldCIsIGRlZmF1bHQ9ImRlcm1hbW5pc3QiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCBkZWZhdWx0PSJyZXNuZXQxOCIsIGNob2ljZXM9WyJyZXNuZXQxOCIsICJyZXNuZXQ1MCJdKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTI4LCBjaG9pY2VzPVsyOCwgMjI0XSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD0wKQogICAgcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAwKQogICAgcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTEyOCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWxyIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xZS0zKQogICAgcC5hZGRfYXJndW1lbnQoIi0tYW1wIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW5vLWRldGVybWluaXN0aWMiLCBkZXN0PSJkZXRlcm1pbmlzdGljIiwgYWN0aW9uPSJzdG9yZV9mYWxzZSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS13aWR0aC1tdWx0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xLjApCiAgICBwLmFkZF9hcmd1bWVudCgiLS13ZWlnaHRlZC1zYW1wbGVyIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXdlaWdodGVkLWxvc3MiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbnVtLXdvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgcC5hZGRfYXJndW1lbnQoIi0tY2twdC1ldmVyeSIsIHR5cGU9aW50LCBkZWZhdWx0PTUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1uby1yZXN1bWUiLCBkZXN0PSJyZXN1bWUiLCBhY3Rpb249InN0b3JlX2ZhbHNlIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW5vLWRvd25sb2FkIiwgZGVzdD0iZG93bmxvYWQiLCBhY3Rpb249InN0b3JlX2ZhbHNlIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1yZXN1bHRzLWRpciIsIGRlZmF1bHQ9InJlc3VsdHMiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tdGFnIiwgZGVmYXVsdD0iIikKICAgIHJldHVybiBwLnBhcnNlX2FyZ3MoYXJndikKCgpkZWYgbWFpbihhcmd2PU5vbmUpOgogICAgYXJncyA9IF9wYXJzZV9hcmdzKGFyZ3YpCiAgICBjZmcgPSBSdW5Db25maWcoKip2YXJzKGFyZ3MpKQogICAgcnVuX2NvbmZpZyhjZmcpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=',
    'train.py': 'IiIiVHJhaW5pbmcgbG9vcDogc2VlZGluZywgYmVzdC12YWwtQVVDIGNoZWNrcG9pbnRpbmcsIHJlc3VtZSwgb3B0aW9uYWwgQU1QLgoKUHJvdG9jb2wgKE1lZE1OSVNUIHYyIGJhc2VsaW5lKToKICAgIEFkYW0obHI9MWUtMyksIE11bHRpU3RlcExSKGdhbW1hPTAuMSwgbWlsZXN0b25lcz1bMC41RSwgMC43NUVdKSwKICAgIGJhdGNoIDEyOCwgMTAwIGVwb2NocywgQ3Jvc3NFbnRyb3B5TG9zcyAoQkNFV2l0aExvZ2l0cyBmb3IgbXVsdGktbGFiZWwpLAogICAgbm8gYXVnbWVudGF0aW9uLiBNb2RlbCBzZWxlY3Rpb24gPSBjaGVja3BvaW50IHdpdGggdGhlIGhpZ2hlc3QgdmFsaWRhdGlvbgogICAgbWFjcm8tQVVDIHNlZW4gc28gZmFyOyB0aGF0IGNoZWNrcG9pbnQncyB0ZXN0IEFVQy9BQ0MgaXMgcmVwb3J0ZWQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCB0aW1lCmltcG9ydCByYW5kb20KaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KCmZyb20gLmV2YWx1YXRlIGltcG9ydCBldmFsdWF0ZV9zcGxpdCwgbG9hZF9jaGVja3BvaW50CgoKZGVmIHNldF9zZWVkKHNlZWQsIGRldGVybWluaXN0aWM9VHJ1ZSk6CiAgICAiIiJTZWVkIHB5dGhvbiAvIG51bXB5IC8gdG9yY2ggLyBDVURBLiBSZXR1cm5zIHdoZXRoZXIgY3Vkbm4gaXMgZGV0ZXJtaW5pc3RpYy4iIiIKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgdG9yY2gubWFudWFsX3NlZWQoc2VlZCkKICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNlZWQpCiAgICBpZiBkZXRlcm1pbmlzdGljOgogICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmRldGVybWluaXN0aWMgPSBUcnVlCiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gRmFsc2UKICAgIGVsc2U6CiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uZGV0ZXJtaW5pc3RpYyA9IEZhbHNlCiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gVHJ1ZQogICAgcmV0dXJuIGRldGVybWluaXN0aWMKCgpkZWYgbWFrZV9jcml0ZXJpb24odGFzaywgY2xhc3Nfd2VpZ2h0PU5vbmUsIGRldmljZT1Ob25lKToKICAgIGlmIHRhc2sgPT0gIm11bHRpLWxhYmVsLCBiaW5hcnktY2xhc3MiOgogICAgICAgIHJldHVybiBubi5CQ0VXaXRoTG9naXRzTG9zcygpCiAgICB3ID0gY2xhc3Nfd2VpZ2h0LnRvKGRldmljZSkgaWYgKGNsYXNzX3dlaWdodCBpcyBub3QgTm9uZSBhbmQgZGV2aWNlIGlzIG5vdCBOb25lKSBlbHNlIGNsYXNzX3dlaWdodAogICAgcmV0dXJuIG5uLkNyb3NzRW50cm9weUxvc3Mod2VpZ2h0PXcpCgoKZGVmIF90YXJnZXRzX2Zvcl9sb3NzKHksIHRhc2ssIGRldmljZSk6CiAgICBpZiB0YXNrID09ICJtdWx0aS1sYWJlbCwgYmluYXJ5LWNsYXNzIjoKICAgICAgICByZXR1cm4geS50byh0b3JjaC5mbG9hdDMyKS50byhkZXZpY2UpCiAgICByZXR1cm4geS5zcXVlZXplKGRpbT0xKS5sb25nKCkudG8oZGV2aWNlKSBpZiB5Lm5kaW0gPT0gMiBlbHNlIHkubG9uZygpLnRvKGRldmljZSkKCgpkZWYgdHJhaW5fb25lX2Vwb2NoKG1vZGVsLCBsb2FkZXIsIGNyaXRlcmlvbiwgb3B0aW1pemVyLCB0YXNrLCBkZXZpY2UsCiAgICAgICAgICAgICAgICAgICAgc2NhbGVyPU5vbmUsIHVzZV9hbXA9RmFsc2UpOgogICAgbW9kZWwudHJhaW4oKQogICAgcnVubmluZywgbl9pbWdzLCB0MCA9IDAuMCwgMCwgdGltZS50aW1lKCkKICAgIGZvciB4LCB5IGluIGxvYWRlcjoKICAgICAgICB4ID0geC50byhkZXZpY2UsIG5vbl9ibG9ja2luZz1UcnVlKQogICAgICAgIHRhcmdldCA9IF90YXJnZXRzX2Zvcl9sb3NzKHksIHRhc2ssIGRldmljZSkKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgd2l0aCB0b3JjaC5jdWRhLmFtcC5hdXRvY2FzdChlbmFibGVkPXVzZV9hbXApOgogICAgICAgICAgICBsb2dpdHMgPSBtb2RlbCh4KQogICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKGxvZ2l0cywgdGFyZ2V0KQogICAgICAgIGlmIHVzZV9hbXA6CiAgICAgICAgICAgIHNjYWxlci5zY2FsZShsb3NzKS5iYWNrd2FyZCgpCiAgICAgICAgICAgIHNjYWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgc2NhbGVyLnVwZGF0ZSgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIG9wdGltaXplci5zdGVwKCkKICAgICAgICBydW5uaW5nICs9IGxvc3MuaXRlbSgpICogeC5zaXplKDApCiAgICAgICAgbl9pbWdzICs9IHguc2l6ZSgwKQogICAgZHQgPSB0aW1lLnRpbWUoKSAtIHQwCiAgICByZXR1cm4gcnVubmluZyAvIG1heChuX2ltZ3MsIDEpLCBuX2ltZ3MgLyBtYXgoZHQsIDFlLTkpLCBkdAoKCmRlZiBydW5fdHJhaW5pbmcobW9kZWwsIGxvYWRlcnMsIHRhc2ssICosIGVwb2Nocz0xMDAsIGxyPTFlLTMsIGRldmljZT0iY3VkYSIsCiAgICAgICAgICAgICAgICAgY2xhc3Nfd2VpZ2h0PU5vbmUsIHVzZV9hbXA9RmFsc2UsIGRldGVybWluaXN0aWM9VHJ1ZSwgc2VlZD0wLAogICAgICAgICAgICAgICAgIGNrcHRfZGlyPU5vbmUsIGNrcHRfZXZlcnk9NSwgcmVzdW1lPVRydWUsIGxvZ19mbj1wcmludCk6CiAgICAiIiJUcmFpbiwgc2VsZWN0aW5nIHRoZSBiZXN0LXZhbC1BVUMgY2hlY2twb2ludC4gUmVzdW1hYmxlLgoKICAgIFJldHVybnMgYSBoaXN0b3J5L3Jlc3VsdCBkaWN0LiBXcml0ZXMgYGBiZXN0X21vZGVsLnB0aGBgIGFuZCBwZXJpb2RpYwogICAgYGBsYXN0LnB0aGBgIGludG8gYGBja3B0X2RpcmBgIHNvIGEgcnVuIGNhbiBjb250aW51ZSBpbiBhIGZyZXNoIEthZ2dsZQogICAgc2Vzc2lvbiB2aWEgYGByZXN1bWU9VHJ1ZWBgLgogICAgIiIiCiAgICB0cmFpbl9sb2FkZXIsIHZhbF9sb2FkZXIsIHRlc3RfbG9hZGVyID0gbG9hZGVycwogICAgbW9kZWwgPSBtb2RlbC50byhkZXZpY2UpCgogICAgY3JpdGVyaW9uID0gbWFrZV9jcml0ZXJpb24odGFzaywgY2xhc3Nfd2VpZ2h0PWNsYXNzX3dlaWdodCwgZGV2aWNlPWRldmljZSkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj1scikKICAgIG1pbGVzdG9uZXMgPSBbaW50KDAuNSAqIGVwb2NocyksIGludCgwLjc1ICogZXBvY2hzKV0KICAgIHNjaGVkdWxlciA9IHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5NdWx0aVN0ZXBMUihvcHRpbWl6ZXIsIG1pbGVzdG9uZXM9bWlsZXN0b25lcywgZ2FtbWE9MC4xKQogICAgc2NhbGVyID0gdG9yY2guY3VkYS5hbXAuR3JhZFNjYWxlcihlbmFibGVkPXVzZV9hbXApCgogICAgc3RhcnRfZXBvY2ggPSAwCiAgICBiZXN0X3ZhbF9hdWMgPSAtMS4wCiAgICBiZXN0X2Vwb2NoID0gLTEKICAgIGhpc3RvcnkgPSBbXQoKICAgIGxhc3RfcGF0aCA9IG9zLnBhdGguam9pbihja3B0X2RpciwgImxhc3QucHRoIikgaWYgY2twdF9kaXIgZWxzZSBOb25lCiAgICBiZXN0X3BhdGggPSBvcy5wYXRoLmpvaW4oY2twdF9kaXIsICJiZXN0X21vZGVsLnB0aCIpIGlmIGNrcHRfZGlyIGVsc2UgTm9uZQogICAgaWYgY2twdF9kaXI6CiAgICAgICAgb3MubWFrZWRpcnMoY2twdF9kaXIsIGV4aXN0X29rPVRydWUpCgogICAgaWYgcmVzdW1lIGFuZCBsYXN0X3BhdGggYW5kIG9zLnBhdGguZXhpc3RzKGxhc3RfcGF0aCk6CiAgICAgICAgY2twdCA9IGxvYWRfY2hlY2twb2ludChsYXN0X3BhdGgsIG1vZGVsLCBvcHRpbWl6ZXIsIHNjaGVkdWxlciwgc2NhbGVyLCBtYXBfbG9jYXRpb249ZGV2aWNlKQogICAgICAgIHN0YXJ0X2Vwb2NoID0gY2twdFsiZXBvY2giXSArIDEKICAgICAgICBiZXN0X3ZhbF9hdWMgPSBja3B0LmdldCgiYmVzdF92YWxfYXVjIiwgLTEuMCkKICAgICAgICBiZXN0X2Vwb2NoID0gY2twdC5nZXQoImJlc3RfZXBvY2giLCAtMSkKICAgICAgICBoaXN0b3J5ID0gY2twdC5nZXQoImhpc3RvcnkiLCBbXSkKICAgICAgICBsb2dfZm4oZiJbcmVzdW1lXSBjb250aW51aW5nIGZyb20gZXBvY2gge3N0YXJ0X2Vwb2NofSAoYmVzdF92YWxfYXVjPXtiZXN0X3ZhbF9hdWM6LjRmfSkiKQoKICAgIGZvciBlcG9jaCBpbiByYW5nZShzdGFydF9lcG9jaCwgZXBvY2hzKToKICAgICAgICBsb3NzLCBpcHMsIGR0ID0gdHJhaW5fb25lX2Vwb2NoKAogICAgICAgICAgICBtb2RlbCwgdHJhaW5fbG9hZGVyLCBjcml0ZXJpb24sIG9wdGltaXplciwgdGFzaywgZGV2aWNlLCBzY2FsZXIsIHVzZV9hbXApCiAgICAgICAgc2NoZWR1bGVyLnN0ZXAoKQoKICAgICAgICB2YWxfYXVjLCB2YWxfYWNjLCBfLCBfID0gZXZhbHVhdGVfc3BsaXQobW9kZWwsIHZhbF9sb2FkZXIsIHRhc2ssIGRldmljZSwgdXNlX2FtcCkKICAgICAgICBpc19iZXN0ID0gdmFsX2F1YyA+IGJlc3RfdmFsX2F1YwogICAgICAgIGlmIGlzX2Jlc3Q6CiAgICAgICAgICAgIGJlc3RfdmFsX2F1YywgYmVzdF9lcG9jaCA9IHZhbF9hdWMsIGVwb2NoCiAgICAgICAgICAgIGlmIGJlc3RfcGF0aDoKICAgICAgICAgICAgICAgIHRvcmNoLnNhdmUoeyJtb2RlbCI6IG1vZGVsLnN0YXRlX2RpY3QoKSwgImVwb2NoIjogZXBvY2gsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAidmFsX2F1YyI6IHZhbF9hdWMsICJ2YWxfYWNjIjogdmFsX2FjY30sIGJlc3RfcGF0aCkKCiAgICAgICAgaGlzdG9yeS5hcHBlbmQoZGljdChlcG9jaD1lcG9jaCwgdHJhaW5fbG9zcz1sb3NzLCB2YWxfYXVjPXZhbF9hdWMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB2YWxfYWNjPXZhbF9hY2MsIGltZ3NfcGVyX3NlYz1pcHMsIGVwb2NoX3RpbWVfcz1kdCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxyPW9wdGltaXplci5wYXJhbV9ncm91cHNbMF1bImxyIl0pKQogICAgICAgIGxvZ19mbihmIltle2Vwb2NoOjAzZH1dIGxvc3M9e2xvc3M6LjRmfSB2YWxfYXVjPXt2YWxfYXVjOi40Zn0gIgogICAgICAgICAgICAgICBmInZhbF9hY2M9e3ZhbF9hY2M6LjRmfSB7aXBzOi4wZn0gaW1nL3Mge2R0Oi4xZn1zIgogICAgICAgICAgICAgICArICgiICAqYmVzdCoiIGlmIGlzX2Jlc3QgZWxzZSAiIikpCgogICAgICAgIGlmIGxhc3RfcGF0aCBhbmQgKChlcG9jaCArIDEpICUgY2twdF9ldmVyeSA9PSAwIG9yIGVwb2NoID09IGVwb2NocyAtIDEpOgogICAgICAgICAgICB0b3JjaC5zYXZlKHsibW9kZWwiOiBtb2RlbC5zdGF0ZV9kaWN0KCksICJvcHRpbWl6ZXIiOiBvcHRpbWl6ZXIuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAic2NoZWR1bGVyIjogc2NoZWR1bGVyLnN0YXRlX2RpY3QoKSwgInNjYWxlciI6IHNjYWxlci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICAgICAgICAgICAgICJlcG9jaCI6IGVwb2NoLCAiYmVzdF92YWxfYXVjIjogYmVzdF92YWxfYXVjLAogICAgICAgICAgICAgICAgICAgICAgICAiYmVzdF9lcG9jaCI6IGJlc3RfZXBvY2gsICJoaXN0b3J5IjogaGlzdG9yeX0sIGxhc3RfcGF0aCkKCiAgICAjIFJlbG9hZCBiZXN0IGNoZWNrcG9pbnQgZm9yIGZpbmFsIHJlcG9ydGluZy4KICAgIGlmIGJlc3RfcGF0aCBhbmQgb3MucGF0aC5leGlzdHMoYmVzdF9wYXRoKToKICAgICAgICBsb2FkX2NoZWNrcG9pbnQoYmVzdF9wYXRoLCBtb2RlbCwgbWFwX2xvY2F0aW9uPWRldmljZSkKCiAgICB0cmFpbl9hdWMsIHRyYWluX2FjYywgXywgXyA9IGV2YWx1YXRlX3NwbGl0KG1vZGVsLCB0cmFpbl9sb2FkZXIsIHRhc2ssIGRldmljZSwgdXNlX2FtcCkKICAgIHZhbF9hdWMsIHZhbF9hY2MsIF8sIF8gPSBldmFsdWF0ZV9zcGxpdChtb2RlbCwgdmFsX2xvYWRlciwgdGFzaywgZGV2aWNlLCB1c2VfYW1wKQogICAgdGVzdF9hdWMsIHRlc3RfYWNjLCB5X3RydWUsIHlfc2NvcmUgPSBldmFsdWF0ZV9zcGxpdChtb2RlbCwgdGVzdF9sb2FkZXIsIHRhc2ssIGRldmljZSwgdXNlX2FtcCkKCiAgICByZXR1cm4gZGljdCgKICAgICAgICBiZXN0X2Vwb2NoPWJlc3RfZXBvY2gsIGJlc3RfdmFsX2F1Yz1iZXN0X3ZhbF9hdWMsCiAgICAgICAgdHJhaW5fYXVjPXRyYWluX2F1YywgdHJhaW5fYWNjPXRyYWluX2FjYywKICAgICAgICB2YWxfYXVjPXZhbF9hdWMsIHZhbF9hY2M9dmFsX2FjYywKICAgICAgICB0ZXN0X2F1Yz10ZXN0X2F1YywgdGVzdF9hY2M9dGVzdF9hY2MsCiAgICAgICAgaGlzdG9yeT1oaXN0b3J5LCB0ZXN0X3lfdHJ1ZT15X3RydWUsIHRlc3RfeV9zY29yZT15X3Njb3JlLAogICAgKQo=',
}

import base64, os, importlib, sys
os.makedirs("src", exist_ok=True)
for _name, _b64 in SRC_B64.items():
    with open(os.path.join("src", _name), "wb") as _f:
        _f.write(base64.b64decode(_b64))
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
import src
print("wrote src package:", sorted(SRC_B64))
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))


## 2. Metric agreement test (our AUC/ACC vs `medmnist.Evaluator`, tol 1e-6)

In [ ]:
from src import metrics_test
metrics_test.main()   # raises if our metrics drift from the medmnist oracle


## 3. CONFIG — edit this cell each session

`MODE` selects what runs. In `baselines` mode, pick the `TIERS` and a
`MAX_MINUTES` budget you can complete before Kaggle's cap.

In [ ]:
import os

MODE = "baselines"          # "baselines" | "extensions" | "report"

# --- baselines-mode knobs ---
# TIERS is a subset of [1, 2, 3]:
#   T1 = DermaMNIST, 4 configs (R18/R50 x 28/224), 3 seeds        -> 12 runs
#   T2 = 7 small/medium datasets, R18 @ 28, 3 seeds               -> 21 runs
#   T3 = 4 large datasets (Tissue/OCT/Path/Chest), R18 @ 28, 1 seed -> 4 runs
TIERS = [1]
SEEDS = [0, 1, 2]           # T3 always uses seed 0 only, regardless of this
EPOCHS = 100
USE_AMP = False             # True ~halves 224 runtime; flagged in run.json if used
DETERMINISTIC = True
MAX_MINUTES = 480           # stop STARTING new runs after this many minutes (~8h < 9h cap)

# --- persistence for resume across sessions ---
WORK_RESULTS = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
PREV_RESULTS = None         # e.g. "/kaggle/input/medmnist-replication-vN/results" (previous output)

os.makedirs(WORK_RESULTS, exist_ok=True)
if PREV_RESULTS and os.path.isdir(PREV_RESULTS):
    import shutil
    for name in os.listdir(PREV_RESULTS):
        src_d = os.path.join(PREV_RESULTS, name)
        dst_d = os.path.join(WORK_RESULTS, name)
        if os.path.isdir(src_d) and not os.path.exists(dst_d):
            shutil.copytree(src_d, dst_d)
    print("seeded resume state from", PREV_RESULTS, "->", len(os.listdir(WORK_RESULTS)), "run dirs")
print("MODE =", MODE, "| results dir =", WORK_RESULTS)

## 4. Baselines — train the tiered run matrix (resumable)

In [ ]:
import time, json
from src.run import RunConfig, run_config, run_matrix

if MODE == "baselines":
    matrix = [m for m in run_matrix(seeds=tuple(SEEDS)) if m[0] in TIERS]
    print(f"{len(matrix)} runs selected for tiers {TIERS}")
    t_start = time.time()
    for tier, dataset, model, size, seed in matrix:
        cfg = RunConfig(dataset=dataset, model=model, size=size, seed=seed,
                        epochs=EPOCHS, amp=USE_AMP, deterministic=DETERMINISTIC,
                        results_dir=WORK_RESULTS)
        rj = os.path.join(WORK_RESULTS, cfg.run_name(), "run.json")
        if os.path.exists(rj):
            d = json.load(open(rj))
            if len(d.get("history", [])) >= EPOCHS:
                print("skip (done):", cfg.run_name()); continue
        if (time.time() - t_start) / 60 > MAX_MINUTES:
            print("MAX_MINUTES reached; stopping before", cfg.run_name(),
                  "(re-run next session to continue)"); break
        run_config(cfg)
    print("baselines pass complete for this session.")
else:
    print("skipped (MODE != 'baselines')")


## 5. Extensions — equity study on DermaMNIST (R18 @ 28)

Reuses the Prompt-1 models / loaders / metrics / training loop unchanged (only
the sampler, loss weights, or channel widths differ) and keeps everything at
28×28. Produces, from saved artifacts:

- **Per-class breakdown** with ROC-AUC **and PR-AUC (average precision)**,
  precision/recall/F1, carrying ±std across the three DermaMNIST seeds.
- **Frequency vs performance** (Pearson/Spearman): the core equity claim.
- **Bias mitigation** — `WeightedRandomSampler` and inverse-frequency weighted
  loss, 3 seeds each; per-class + aggregate tradeoff.
- **Lightweight 0.5× variant** — params, MB, latency, rare-vs-common degradation.
- **Corruption robustness** (inference-only): Gaussian noise, JPEG, brightness.
- **Confidence & calibration** (inference-only): reliability, ECE, per-class confidence.

CSVs land under `results/extension/`; figures under `results/extension/figures/`
(PDF + 300-dpi PNG) so they survive Kaggle's session split and the report cell
copies them into `report/figures/`. Variant runs carry flags/tags so they never
pollute the baseline comparison table.

In [ ]:
import numpy as np, json, csv, os
from src.run import RunConfig, run_config
from src import extensions as ext, evaluate as ev, data as dm, models as M, metrics as mx
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _run_dir(cfg):
    return os.path.join(WORK_RESULTS, cfg.run_name())

def _done(cfg):
    return os.path.exists(os.path.join(_run_dir(cfg), "best_model.pth"))

def _ensure(cfg):
    if not _done(cfg):
        run_config(cfg)
    return cfg

def load_model(cfg):
    info = dm.get_info(cfg.dataset); nC = len(info["label"])
    model = M.build_model(cfg.model, cfg.size, nC, width_mult=cfg.width_mult).to(DEVICE)
    ev.load_checkpoint(os.path.join(_run_dir(cfg), "best_model.pth"), model, map_location=DEVICE)
    return model

def get_preds(cfg, model=None):
    info = dm.get_info(cfg.dataset)
    if model is None:
        model = load_model(cfg)
    _, _, test_loader = dm.get_loaders(cfg.dataset, cfg.size, batch_size=128,
                                       pin_memory=(DEVICE == "cuda"))
    return ev.predict(model, test_loader, info["task"], DEVICE)

if MODE == "extensions":
    EXT_OUT = os.path.join(WORK_RESULTS, "extension"); os.makedirs(EXT_OUT, exist_ok=True)
    common = dict(dataset="dermamnist", model="resnet18", size=28,
                  epochs=EPOCHS, amp=USE_AMP, results_dir=WORK_RESULTS)

    # --- ensure the runs we need (baseline seeds reuse Tier 1; variants are new) ---
    base_cfgs  = [RunConfig(**common, seed=s) for s in SEEDS]
    wsamp_cfgs = [RunConfig(**common, seed=s, weighted_sampler=True) for s in SEEDS]
    wloss_cfgs = [RunConfig(**common, seed=s, weighted_loss=True) for s in SEEDS]
    light_cfgs = [RunConfig(**common, seed=s, width_mult=0.5, tag="light") for s in SEEDS]
    for group in (base_cfgs, wsamp_cfgs, wloss_cfgs, light_cfgs):
        for c in group:
            _ensure(c)

    base_done = [c for c in base_cfgs if _done(c)]
    base_preds = [get_preds(c) for c in base_done]              # list of (yt, ys)
    yt, ys = base_preds[0]                                      # seed-0 baseline

    # (1) per-class breakdown with ROC-AUC + PR-AUC, ±std across seeds
    ms = ext.per_class_multiseed("dermamnist", base_preds)
    ms.to_csv(os.path.join(EXT_OUT, "perclass_multiseed.csv"), index=False)
    ext.per_class_analysis("dermamnist", yt, ys, EXT_OUT, tag="baseline")  # confusion + ROC + CSV
    ext.plot_pr_curves("dermamnist", yt, ys, EXT_OUT)
    ext.plot_per_class_performance("dermamnist", ms, EXT_OUT)
    ext.plot_class_distribution("dermamnist", EXT_OUT)
    print("per-class (baseline, mean over %d seeds):" % len(base_preds))
    print(ms.to_string(index=False))

    # (2) frequency vs performance
    freq = ext.frequency_performance("dermamnist", ms, EXT_OUT)
    print("\nfreq vs recall: Pearson r=%.3f  Spearman rho=%.3f  (AUC Pearson=%.3f)"
          % (freq["pearson_recall"], freq["spearman_recall"], freq["pearson_auc"]))

    # (3) bias mitigation: per-class (multiseed) + aggregate tradeoff
    wsamp_preds = [get_preds(c) for c in wsamp_cfgs if _done(c)]
    wloss_preds = [get_preds(c) for c in wloss_cfgs if _done(c)]
    variants = {"baseline": (yt, ys),
                "weighted_sampler": wsamp_preds[0] if wsamp_preds else (yt, ys),
                "weighted_loss": wloss_preds[0] if wloss_preds else (yt, ys)}
    summary, _ = ext.bias_comparison("dermamnist", variants, EXT_OUT)
    mit_tables = {"baseline": ms,
                  "weighted_sampler": ext.per_class_multiseed("dermamnist", wsamp_preds) if wsamp_preds else ms,
                  "weighted_loss": ext.per_class_multiseed("dermamnist", wloss_preds) if wloss_preds else ms}
    ext.plot_mitigation("dermamnist", mit_tables, EXT_OUT)
    print("\nbias mitigation summary (equity vs accuracy tradeoff):")
    print(summary.to_string(index=False))

    # (4) lightweight variant: profile, efficiency tradeoff, rare-vs-common
    light_done = [c for c in light_cfgs if _done(c)]
    ylt, yls = get_preds(light_done[0]) if light_done else (yt, ys)
    light_tbl = ext.per_class_table("dermamnist", ylt, yls)
    base_tbl = ext.per_class_table("dermamnist", yt, ys)
    merged, corr = ext.rare_vs_common_degradation("dermamnist", base_tbl, light_tbl, EXT_OUT)
    info = dm.get_info("dermamnist"); nC = len(info["label"])
    prof_full  = ext.profile_model(M.build_model("resnet18", 28, nC, width_mult=1.0), DEVICE, size=28)
    prof_light = ext.profile_model(M.build_model("resnet18", 28, nC, width_mult=0.5), DEVICE, size=28)
    profiles = {"resnet18_full": prof_full, "resnet18_0.5x": prof_light}
    eff_metrics = {"resnet18_full": mx.evaluate(yt, ys, info["task"]),
                   "resnet18_0.5x": mx.evaluate(ylt, yls, info["task"])}
    ext.plot_efficiency(profiles, eff_metrics, EXT_OUT)
    with open(os.path.join(EXT_OUT, "model_size_inference.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["variant", "n_params", "size_mb", "latency_ms_per_image", "test_auc", "test_acc"])
        for name, prof, (a, b) in [("resnet18_full", prof_full, eff_metrics["resnet18_full"]),
                                   ("resnet18_0.5x", prof_light, eff_metrics["resnet18_0.5x"])]:
            w.writerow([name, prof["n_params"], f"{prof['size_mb']:.3f}",
                        f"{prof['latency_ms_per_image']:.4f}", f"{a:.4f}", f"{b:.4f}"])

    # (5) corruption robustness (inference-only) on the baseline model
    base_model = load_model(base_done[0])
    test_imgs = dm.get_dataset("dermamnist", "test", 28).imgs   # (N,28,28,3) uint8, aligned with yt
    rob = ext.robustness_eval(base_model, "dermamnist", test_imgs, yt, device=DEVICE)
    rob.to_csv(os.path.join(EXT_OUT, "robustness.csv"), index=False)
    ext.plot_robustness(rob, EXT_OUT)

    # (6) confidence & calibration (inference-only)
    ext.calibration_analysis("dermamnist", yt, ys, out_dir=EXT_OUT)  # writes calibration_ece.csv
    ext.plot_calibration("dermamnist", yt, ys, EXT_OUT)

    # misclassification gallery (rare-class errors)
    ext.plot_misclassified_gallery("dermamnist", yt, ys, test_imgs, EXT_OUT)

    # short equity writeup
    REPORT = "/kaggle/working/report" if os.path.isdir("/kaggle/working") else "report"
    os.makedirs(REPORT, exist_ok=True)
    with open(os.path.join(REPORT, "extension.md"), "w") as f:
        f.write("# DermaMNIST equity & extension study\n\n")
        f.write("ResNet-18 @ 28, %d baseline seed(s). Kept separate from the replication's comparison.md.\n\n" % len(base_preds))
        f.write("## Frequency vs performance\n")
        f.write("- Pearson r (count vs recall) = %.3f; Spearman rho = %.3f; Pearson (count vs AUC) = %.3f.\n\n" % (
            freq["pearson_recall"], freq["spearman_recall"], freq["pearson_auc"]))
        f.write("## Bias mitigation (equity vs accuracy tradeoff)\n\n")
        try:
            f.write(summary.to_markdown(index=False) + "\n\n")
        except Exception:  # tabulate not present -> plain text fallback
            f.write("```\n" + summary.to_string(index=False) + "\n```\n\n")
        f.write("## Lightweight variant\n")
        f.write("- params %.2fM -> %.2fM; latency %.3f -> %.3f ms/img; freq~F1-drop corr = %.3f.\n" % (
            prof_full["n_params"]/1e6, prof_light["n_params"]/1e6,
            prof_full["latency_ms_per_image"], prof_light["latency_ms_per_image"], corr))
    print("\nextension artifacts ->", EXT_OUT, "| figures ->", os.path.join(EXT_OUT, "figures"))
else:
    print("skipped (MODE != 'extensions')")

## 6. Report — aggregate table + every figure for the paper

In [ ]:
if MODE == "report":
    from src import aggregate as agg
    from src import figures_replication as figrep
    from IPython.display import Image, display
    import shutil, glob

    REPORT = "/kaggle/working/report" if os.path.isdir("/kaggle/working") else "report"
    FIGDIR = os.path.join(REPORT, "figures")
    os.makedirs(FIGDIR, exist_ok=True)

    # 1. Aggregate every baseline run -> comparison.{csv,md} vs paper Table 3.
    rows = agg.aggregate(WORK_RESULTS, REPORT)
    print(open(os.path.join(REPORT, "comparison.md")).read())

    # 2. ReScience figures, generated from results/ alone into report/figures/
    #    (PDF + 300-dpi PNG each). Never re-runs training.
    fig_pngs = figrep.generate_all(WORK_RESULTS, REPORT)

    # 3. Copy the extension figures/CSVs (from results/extension) into the report.
    EXT_OUT = os.path.join(WORK_RESULTS, "extension")
    ext_pngs = []
    if os.path.isdir(EXT_OUT):
        for p in glob.glob(os.path.join(EXT_OUT, "figures", "*.png")) + glob.glob(os.path.join(EXT_OUT, "*.png")):
            dst = os.path.join(FIGDIR, os.path.basename(p)); shutil.copy(p, dst); ext_pngs.append(dst)
        for p in glob.glob(os.path.join(EXT_OUT, "figures", "*.pdf")):
            shutil.copy(p, os.path.join(FIGDIR, os.path.basename(p)))
        for p in glob.glob(os.path.join(EXT_OUT, "*.csv")):
            shutil.copy(p, os.path.join(REPORT, os.path.basename(p)))

    print("\nfigures written to", FIGDIR)
    for p in fig_pngs + sorted(set(ext_pngs)):
        print("---", os.path.basename(p))
        display(Image(p))
else:
    print("skipped (MODE != 'report')")

## Outputs

Everything lands under `/kaggle/working/` and is saved with the notebook version:

- `results/<run_name>/` — `run.json` (config + metrics + pinned versions + seed +
  wall-clock + **peak GPU memory**), `best_model.pth`, `last.pth`, and the
  medmnist-convention predictions CSV, e.g.
  `dermamnist_test_[AUC]0.917_[ACC]0.735@seed0.csv`.
- `results/extension/` — equity CSVs (`perclass_multiseed.csv`, `bias_summary.csv`,
  `model_size_inference.csv`, `robustness.csv`, `calibration_ece.csv`,
  `lightweight_degradation.csv`) and `figures/` (PDF + 300-dpi PNG).
- `report/comparison.{csv,md}` — our mean±std vs paper Table 3 (all twelve R18@28
  datasets + the DermaMNIST four-config matrix), with signed deltas and flags.
- `report/extension.md` — equity findings + mitigation tradeoff, kept separate.
- `report/figures/` — every figure as **PDF + 300-dpi PNG**:
  - *Replication:* `fig1_reproduction` (ours-vs-paper AUC/ACC scatter + Derma 4-config bars),
    `fig2_training_curves` (LR-decay markers), `fig3_seed_variance`,
    `fig4_delta_heatmap`, `fig5_compute_footprint` (epoch time / peak GPU memory).
  - *Equity:* `ext_class_distribution`, `ext_per_class_perf`, `ext_freq_perf`,
    `confusion_baseline`, `roc_baseline`, `ext_pr_curves`, `bias_perclass_f1`,
    `ext_mitigation`, `ext_efficiency`, `ext_robustness`, `ext_calibration`,
    `ext_misclassified`.

To continue training next session, attach this notebook's output as input data
and set `PREV_RESULTS` in the CONFIG cell.